# Vault of vaults parameter search

- Search different portfolio weighting methods and historical return lookbacks to see what kind of different vault portfolios we can construct

# Set up

Set up Trading Strategy data client.


In [ ]:
import logging

from tradingstrategy.client import Client
from tradeexecutor.utils.notebook import setup_charting_and_output, OutputMode, set_notebook_logging

client = Client.create_jupyter_client()

# Set up drawing charts in interactive vector output mode.
# This is slower. See the alternative commented option below.
# setup_charting_and_output(OutputMode.interactive)

# Set up rendering static PNG images.
# This is much faster but disables zoom on any chart.
setup_charting_and_output(OutputMode.static, image_format="png", width=1500, height=1000)


logger = logging.getLogger("strategy")



## Chain configuration

- Choose target chains and their vults

In [ ]:
from eth_defi.token import USDC_NATIVE_TOKEN
from eth_defi.token import USDT_NATIVE_TOKEN
from eth_defi.token import WRAPPED_NATIVE_TOKEN

from tradingstrategy.chain import ChainId
from tradingstrategy.lending import LendingProtocolType

CHAIN_ID = ChainId.cross_chain
PRIMARY_CHAIN_ID = ChainId.arbitrum
HYPERCORE_CHAIN_ID = ChainId.hypercore


# We define our main trading universe,
# and then Ethereum mainnet as a validation set
match CHAIN_ID:

    case ChainId.cross_chain:
        # Special backtest for pairs across all chains what a cross chain strategy would look lijke

        EXCHANGES = ("uniswap-v2", "uniswap-v3")
        SUPPORTING_PAIRS = [
            (ChainId.arbitrum, "uniswap-v3", "WETH", "USDC", 0.0005),
            (ChainId.ethereum, "uniswap-v3", "WETH", "USDC", 0.0005),
            (ChainId.ethereum, "uniswap-v3", "WBTC", "USDC", 0.003),
        ]
        LENDING_RESERVES = None
        PREFERRED_STABLECOIN = USDC_NATIVE_TOKEN[PRIMARY_CHAIN_ID].lower()


        # Data source: https://top-defi-vaults.tradingstrategy.ai/top_vaults_by_chain.json
        # Updated: 2026-03-11
        # Denomination filter: USDC, USDC.e, crvUSD, USDS, USDT (and variants USD₮0, USDt, USDT0)
        # Excluded: risk=Blacklisted or risk=Dangerous
        # Min TVL: $50k
        # Min age: 0.3y (except Monad 0.1y)
        # Sorted by 1y CAGR, top 20 per chain (top 60 for Hypercore)
        VAULTS = [
            #
            # Ethereum
            #
            # 16 vaults, sorted by 1y CAGR
            # Filter: min TVL $50k, min age 0.3y, denomination in (USDC, USDC.e, crvUSD, USDS, USDT/USD₮0), exclude Blacklisted/Dangerous
            #

            # [USDS] Pocky's High Yield USDS (protocol=morpho, age=0.4y, CAGR 1m=11.07%, CAGR 3m=31.25%, CAGR 1y=40.21%, CAGR all=40.21%, TVL=$233k)
            (ChainId.ethereum, "0x2e87d6bfa3f2a932e0c70a32607c0b839404984d"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x2e87d6bfa3f2a932e0c70a32607c0b839404984d

            # [USDC] cSuperior Quality Private Credit USDC (protocol=csigma-finance, age=1.3y, CAGR 1m=15.15%, CAGR 3m=15.65%, CAGR 1y=36.45%, CAGR all=34.63%, TVL=$401k)
            (ChainId.ethereum, "0x438982ea288763370946625fd76c2508ee1fb229"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x438982ea288763370946625fd76c2508ee1fb229

            # [USDT] cSuperior Quality Private Credit (protocol=csigma-finance, age=1.4y, CAGR 1m=17.35%, CAGR 3m=17.60%, CAGR 1y=31.70%, CAGR all=30.20%, TVL=$55k)
            (ChainId.ethereum, "0x50d59b785df23728d9948804f8ca3543237a1495"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x50d59b785df23728d9948804f8ca3543237a1495

            # [USDS] Longtail Strategy USDS (protocol=morpho, age=0.4y, CAGR 1m=0.04%, CAGR 3m=2.37%, CAGR 1y=22.36%, CAGR all=22.36%, TVL=$369k)
            (ChainId.ethereum, "0x786977528b0265c5c5bc9544ac56c863c03e34d1"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x786977528b0265c5c5bc9544ac56c863c03e34d1

            # [USDC] Everstone (protocol=morpho, age=0.3y, CAGR 1m=6.96%, CAGR 3m=8.36%, CAGR 1y=22.12%, CAGR all=22.12%, TVL=$389k)
            (ChainId.ethereum, "0x09c4c7b1d2e9aa7506db8b76f1dbbd61c08c114b"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x09c4c7b1d2e9aa7506db8b76f1dbbd61c08c114b

            # [USDT] DeltaUSD HyperLiquid USDN Funding Arb (protocol=lagoon-finance, age=0.6y, CAGR 1m=13.85%, CAGR 3m=12.89%, CAGR 1y=20.55%, CAGR all=20.55%, TVL=$598k)
            (ChainId.ethereum, "0x01f461a0bbb218bc1943aa027c5bbc424391e541"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x01f461a0bbb218bc1943aa027c5bbc424391e541

            # [USDS] Nucleon USDS (protocol=morpho, age=0.4y, CAGR 1m=0.01%, CAGR 3m=0.00%, CAGR 1y=20.27%, CAGR all=20.27%, TVL=$150k)
            (ChainId.ethereum, "0xedc72b49542e4362c677b8369bc23882ed635a75"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xedc72b49542e4362c677b8369bc23882ed635a75

            # [USDC] Hub Capital USDC vault (protocol=lagoon-finance, age=0.5y, CAGR 1m=21.06%, CAGR 3m=16.10%, CAGR 1y=17.44%, CAGR all=17.44%, TVL=$1.0M)
            (ChainId.ethereum, "0xca790385506b790554571cbc9da73f0130cdcfd5"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xca790385506b790554571cbc9da73f0130cdcfd5

            # [USDS] ETH Strategy Perpetual Note (protocol=eth-strategy, age=0.5y, CAGR 1m=14.17%, CAGR 3m=18.40%, CAGR 1y=16.63%, CAGR all=16.63%, TVL=$3.8M)
            (ChainId.ethereum, "0xb250c9e0f7be4cff13f94374c993ac445a1385fe"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xb250c9e0f7be4cff13f94374c993ac445a1385fe

            # [USDC] Syntropia Boosted (protocol=lagoon-finance, age=0.4y, CAGR 1m=6.74%, CAGR 3m=11.43%, CAGR 1y=13.82%, CAGR all=13.82%, TVL=$271k)
            (ChainId.ethereum, "0x8df3deba711ae4a9af16cbca5e4fbb1402f036d5"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x8df3deba711ae4a9af16cbca5e4fbb1402f036d5

            # [USDC] Anchorage/AUTOfinance anchrgUSD (protocol=auto-finance, age=0.3y, CAGR 1m=5.55%, CAGR 3m=6.86%, CAGR 1y=12.70%, CAGR all=12.70%, TVL=$322k)
            (ChainId.ethereum, "0x1abd0403591be494771115d74ed9e120530f356e"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x1abd0403591be494771115d74ed9e120530f356e

            # [USDC] cSigma USD (protocol=csigma-finance, age=0.5y, CAGR 1m=10.41%, CAGR 3m=11.15%, CAGR 1y=12.59%, CAGR all=12.59%, TVL=$51k)
            (ChainId.ethereum, "0xd5d097f278a735d0a3c609deee71234cac14b47e"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xd5d097f278a735d0a3c609deee71234cac14b47e

            # [USDC] Gami USDC (protocol=lagoon-finance, age=0.3y, CAGR 1m=11.73%, CAGR 3m=12.56%, CAGR 1y=12.54%, CAGR all=12.54%, TVL=$1.3M)
            (ChainId.ethereum, "0xdae854d0896ad2fee335689a3f7b4a95fd1a3e46"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xdae854d0896ad2fee335689a3f7b4a95fd1a3e46

            # [USDC] Moon Digital AM USDC (protocol=lagoon-finance, age=0.8y, CAGR 1m=5.30%, CAGR 3m=9.12%, CAGR 1y=11.02%, CAGR all=11.02%, TVL=$794k)
            (ChainId.ethereum, "0xa00f63e85b3d242568a9edecb48f5e2cf879b07b"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xa00f63e85b3d242568a9edecb48f5e2cf879b07b

            # [USDC] Sylva Concentrated Liquidity (protocol=upshift, age=1.0y, CAGR 1m=0.00%, CAGR 3m=0.00%, CAGR 1y=10.89%, CAGR all=10.89%, TVL=$3.0M)
            (ChainId.ethereum, "0xc428439fb7b1efe56360eb837ca98f551fdd9b26"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xc428439fb7b1efe56360eb837ca98f551fdd9b26

            # [USDC] Syntropia USDC Core (protocol=lagoon-finance, age=0.4y, CAGR 1m=5.75%, CAGR 3m=9.16%, CAGR 1y=10.52%, CAGR all=10.52%, TVL=$1.9M)
            (ChainId.ethereum, "0x1b2cb79a4564206f53ba80b4d780f251b4ae6765"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x1b2cb79a4564206f53ba80b4d780f251b4ae6765

            # [USDC] AQRU Maple Pool USDC1 (protocol=maple, age=3.1y, CAGR 1m=17.21%, CAGR 3m=17.15%, CAGR 1y=16.68%, CAGR all=15.29%, TVL=$7.5M)
            # Deposits disabled: Max deposit cap reached (maxDeposit=0)
            (ChainId.ethereum, "0xe9d33286f0e37f517b1204aa6da085564414996d"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xe9d33286f0e37f517b1204aa6da085564414996d

            # [USDS] Teller USDS (protocol=yearn, age=0.4y, CAGR 1m=0.02%, CAGR 3m=1.62%, CAGR 1y=12.41%, CAGR all=12.41%, TVL=$370k)
            # Deposits disabled: Max deposit cap reached (maxDeposit=0)
            (ChainId.ethereum, "0xbfb6a3eabf605a35860ca7e78429e3022c54f236"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xbfb6a3eabf605a35860ca7e78429e3022c54f236

            # [USDC] Kryptonim Line of Credit (protocol=truefi, age=2.3y, CAGR 1m=11.40%, CAGR 3m=11.59%, CAGR 1y=12.23%, CAGR all=13.45%, TVL=$655k)
            # Deposits disabled: Max deposit cap reached (maxDeposit=0)
            (ChainId.ethereum, "0x84ac855f7f423e92e6d0316ec447253732ba4082"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x84ac855f7f423e92e6d0316ec447253732ba4082

            # [USDC] Credit Coop Lending Vault (protocol=unknown, age=0.7y, CAGR 1m=11.04%, CAGR 3m=10.21%, CAGR 1y=10.66%, CAGR all=10.66%, TVL=$6.4M)
            # Deposits disabled: Max deposit cap reached (maxDeposit=0)
            (ChainId.ethereum, "0x6c99a74a62aaf2e6aa3ff08ce7661d5c86e01dbc"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x6c99a74a62aaf2e6aa3ff08ce7661d5c86e01dbc

            #
            # Base
            #
            # 18 vaults, sorted by 1y CAGR
            # Filter: min TVL $50k, min age 0.3y, denomination in (USDC, USDC.e, crvUSD, USDS, USDT/USD₮0), exclude Blacklisted/Dangerous
            #

            # [USDC] Muscadine USDC Vault (protocol=morpho, age=0.7y, CAGR 1m=3.93%, CAGR 3m=4.58%, CAGR 1y=172.17%, CAGR all=172.17%, TVL=$106k)
            (ChainId.base, "0xf7e26fa48a568b8b0038e104dfd8abdf0f99074f"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xf7e26fa48a568b8b0038e104dfd8abdf0f99074f

            # [USDC] Mo Earn Max USDC (protocol=morpho, age=0.4y, CAGR 1m=10.80%, CAGR 3m=12.80%, CAGR 1y=22.42%, CAGR all=22.42%, TVL=$117k)
            (ChainId.base, "0x3094b241aade60f91f1c82b0628a10d9501462f9"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x3094b241aade60f91f1c82b0628a10d9501462f9

            # [USDC] USDC Fluid Lender (protocol=fluid, age=0.5y, CAGR 1m=4.76%, CAGR 3m=4.48%, CAGR 1y=13.27%, CAGR all=13.27%, TVL=$1.5M)
            (ChainId.base, "0x70fffbacb53ef74903ac074aae769414a70970d1"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x70fffbacb53ef74903ac074aae769414a70970d1

            # [USDC] ArcadiaV2 USD Coin Debt (protocol=arcadia-finance, age=2.0y, CAGR 1m=7.76%, CAGR 3m=10.95%, CAGR 1y=12.51%, CAGR all=12.98%, TVL=$1.4M)
            (ChainId.base, "0x3ec4a293fb906dd2cd440c20decb250def141df1"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x3ec4a293fb906dd2cd440c20decb250def141df1

            # [USDC] DeTrade Core USDC (protocol=lagoon-finance, age=1.0y, CAGR 1m=9.38%, CAGR 3m=7.96%, CAGR 1y=11.99%, CAGR all=11.95%, TVL=$655k)
            (ChainId.base, "0x8092ca384d44260ea4feaf7457b629b8dc6f88f0"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x8092ca384d44260ea4feaf7457b629b8dc6f88f0

            # [USDC] Harvest: USDC Vault (0xC777) (protocol=harvest-finance, age=0.8y, CAGR 1m=5.73%, CAGR 3m=6.43%, CAGR 1y=10.72%, CAGR all=10.72%, TVL=$3.9M)
            (ChainId.base, "0xc777031d50f632083be7080e51e390709062263e"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xc777031d50f632083be7080e51e390709062263e

            # [USDC] gTrade (Gains Network USDC) (protocol=gains-network, age=1.5y, CAGR 1m=4.48%, CAGR 3m=6.70%, CAGR 1y=10.29%, CAGR all=12.18%, TVL=$2.4M)
            (ChainId.base, "0xad20523a7dc37babc1cc74897e4977232b3d02e5"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xad20523a7dc37babc1cc74897e4977232b3d02e5

            # [USDC] Wrapped Senior ArcadiaV2 USD Coin (protocol=arcadia-finance, age=1.7y, CAGR 1m=4.33%, CAGR 3m=7.47%, CAGR 1y=8.81%, CAGR all=10.10%, TVL=$368k)
            (ChainId.base, "0xbc10718571fcb3c3f67800e7c0887e450d2ff398"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xbc10718571fcb3c3f67800e7c0887e450d2ff398

            # [USDC] Senior ArcadiaV2 USD Coin (protocol=arcadia-finance, age=2.0y, CAGR 1m=4.31%, CAGR 3m=7.39%, CAGR 1y=8.81%, CAGR all=11.20%, TVL=$2.3M)
            (ChainId.base, "0xefe32813dba3a783059d50e5358b9e3661218dad"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xefe32813dba3a783059d50e5358b9e3661218dad

            # [USDC] GUSDC (protocol=lagoon-finance, age=0.3y, CAGR 1m=7.58%, CAGR 3m=7.26%, CAGR 1y=8.26%, CAGR all=8.26%, TVL=$237k)
            (ChainId.base, "0xd5c22fa3f7ee979ed7c28e36669b29797ab277e4"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xd5c22fa3f7ee979ed7c28e36669b29797ab277e4

            # [USDC] yoVaultUSD (protocol=yo, age=0.9y, CAGR 1m=5.12%, CAGR 3m=5.92%, CAGR 1y=7.92%, CAGR all=7.92%, TVL=$31.4M)
            (ChainId.base, "0x0000000f2eb9f69274678c76222b35eec7588a65"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x0000000f2eb9f69274678c76222b35eec7588a65

            # [USDC] HYPE++ Base (protocol=d2-finance, age=0.4y, CAGR 1m=18.59%, CAGR 3m=10.88%, CAGR 1y=7.88%, CAGR all=7.88%, TVL=$1.5M)
            (ChainId.base, "0x2406aacbdf8463176deb285adaa81768415b6c7e"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x2406aacbdf8463176deb285adaa81768415b6c7e

            # [USDC] 722Capital-USDC (protocol=lagoon-finance, age=0.8y, CAGR 1m=8.74%, CAGR 3m=4.27%, CAGR 1y=7.67%, CAGR all=7.67%, TVL=$1.3M)
            (ChainId.base, "0xb09f761cb13baca8ec087ac476647361b6314f98"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xb09f761cb13baca8ec087ac476647361b6314f98

            # [USDC] yoUSD Looopeer (protocol=ipor-fusion, age=0.5y, CAGR 1m=3.22%, CAGR 3m=4.13%, CAGR 1y=7.53%, CAGR all=7.53%, TVL=$1.8M)
            (ChainId.base, "0x1166250d1d6b5a1dbb73526257f6bb2bbe235295"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x1166250d1d6b5a1dbb73526257f6bb2bbe235295

            # [USDC] Harvest: USDC Vault (0xed5A) (protocol=harvest-finance, age=0.8y, CAGR 1m=4.39%, CAGR 3m=6.74%, CAGR 1y=7.44%, CAGR all=7.44%, TVL=$238k)
            (ChainId.base, "0xed5a6691a3663045c6d9f12f334fbc796ee12f45"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xed5a6691a3663045c6d9f12f334fbc796ee12f45

            # [USDC] Tokemak baseUSD (protocol=auto-finance, age=0.8y, CAGR 1m=4.27%, CAGR 3m=5.28%, CAGR 1y=7.33%, CAGR all=7.33%, TVL=$16.1M)
            (ChainId.base, "0x9c6864105aec23388c89600046213a44c384c831"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x9c6864105aec23388c89600046213a44c384c831

            # [USDC] Autopilot USDC Base (protocol=ipor-fusion, age=1.0y, CAGR 1m=3.48%, CAGR 3m=4.29%, CAGR 1y=6.93%, CAGR all=6.92%, TVL=$780k)
            (ChainId.base, "0x0d877dc7c8fa3ad980dfdb18b48ec9f8768359c4"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x0d877dc7c8fa3ad980dfdb18b48ec9f8768359c4

            # [USDC] Harvest: USDC Vault (0x9061) (protocol=harvest-finance, age=2.4y, CAGR 1m=0.78%, CAGR 3m=3.19%, CAGR 1y=6.73%, CAGR all=11.80%, TVL=$204k)
            (ChainId.base, "0x90613e167d42ca420942082157b42af6fc6a8087"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x90613e167d42ca420942082157b42af6fc6a8087

            # [USDC] Chimi USDC Vault (protocol=yearn, age=0.5y, CAGR 1m=6.35%, CAGR 3m=6.61%, CAGR 1y=8.61%, CAGR all=8.61%, TVL=$83k)
            # Deposits disabled: Max deposit cap reached (maxDeposit=0)
            (ChainId.base, "0xcb2f26898c0893c0bdd5cf76417cf9b2258af0ed"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xcb2f26898c0893c0bdd5cf76417cf9b2258af0ed

            # [USDC] Credit Coop Lending Vault (protocol=unknown, age=1.1y, CAGR 1m=10.30%, CAGR 3m=7.21%, CAGR 1y=8.16%, CAGR all=8.09%, TVL=$72k)
            # Deposits disabled: Max deposit cap reached (maxDeposit=0)
            (ChainId.base, "0x214699b0ad2e26ffef0247fd0c244bb7fedc85ce"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x214699b0ad2e26ffef0247fd0c244bb7fedc85ce

            #
            # Arbitrum
            #
            # 19 vaults, sorted by 1y CAGR
            # Filter: min TVL $50k, min age 0.3y, denomination in (USDC, USDC.e, crvUSD, USDS, USDT/USD₮0), exclude Blacklisted/Dangerous
            #

            # [USDC] HYPE++ (protocol=d2-finance, age=1.3y, CAGR 1m=18.99%, CAGR 3m=12.56%, CAGR 1y=27.68%, CAGR all=33.10%, TVL=$6.0M)
            (ChainId.arbitrum, "0x75288264fdfea8ce68e6d852696ab1ce2f3e5004"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x75288264fdfea8ce68e6d852696ab1ce2f3e5004

            # [USDC] Plutus Hedge Token (protocol=plutus, age=1.1y, CAGR 1m=3.54%, CAGR 3m=10.17%, CAGR 1y=20.96%, CAGR all=18.93%, TVL=$255k)
            (ChainId.arbitrum, "0x58bfc95a864e18e8f3041d2fcd3418f48393fe6a"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x58bfc95a864e18e8f3041d2fcd3418f48393fe6a

            # [USDC] Tokemak arbUSD (protocol=auto-finance, age=0.5y, CAGR 1m=5.13%, CAGR 3m=4.65%, CAGR 1y=20.55%, CAGR all=20.55%, TVL=$1.3M)
            (ChainId.arbitrum, "0xf63b7f49b4f5dc5d0e7e583cfd79dc64e646320c"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xf63b7f49b4f5dc5d0e7e583cfd79dc64e646320c

            # [USDC] Angmar Capital (protocol=lagoon-finance, age=0.4y, CAGR 1m=10.86%, CAGR 3m=16.61%, CAGR 1y=18.55%, CAGR all=18.55%, TVL=$2.7M)
            (ChainId.arbitrum, "0x1723cb57af58efb35a013870c90fcc3d60174a4e"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x1723cb57af58efb35a013870c90fcc3d60174a4e

            # [crvUSD] Llama Lend asdCRV / crvUSD (protocol=llama-lend, age=1.7y, CAGR 1m=14.82%, CAGR 3m=15.42%, CAGR 1y=11.83%, CAGR all=13.32%, TVL=$780k)
            (ChainId.arbitrum, "0xc8248953429d707c6a2815653eca89846ffaa63b"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xc8248953429d707c6a2815653eca89846ffaa63b

            # [USDC] Ostium Liquidity Pool Vault (protocol=ostium, age=1.7y, CAGR 1m=48.37%, CAGR 3m=29.05%, CAGR 1y=11.49%, CAGR all=8.93%, TVL=$57.9M)
            (ChainId.arbitrum, "0x20d419a8e12c45f88fda7c5760bb6923cee27f98"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x20d419a8e12c45f88fda7c5760bb6923cee27f98

            # [USD₮0] Steakhouse High Yield USDT0 (protocol=morpho, age=0.5y, CAGR 1m=3.48%, CAGR 3m=8.20%, CAGR 1y=11.14%, CAGR all=11.14%, TVL=$19.7M)
            (ChainId.arbitrum, "0x4739e2c293bdcd835829aa7c5d7fbdee93565d1a"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x4739e2c293bdcd835829aa7c5d7fbdee93565d1a

            # [USDC] gTrade (Gains Network USDC) (protocol=gains-network, age=2.1y, CAGR 1m=6.33%, CAGR 3m=7.54%, CAGR 1y=9.69%, CAGR all=12.41%, TVL=$13.5M)
            (ChainId.arbitrum, "0xd3443ee1e91af28e5fb858fbd0d72a63ba8046e0"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xd3443ee1e91af28e5fb858fbd0d72a63ba8046e0

            # [crvUSD] Llamarisk crvUSD Vault (protocol=ipor-fusion, age=0.4y, CAGR 1m=6.17%, CAGR 3m=7.38%, CAGR 1y=7.59%, CAGR all=7.59%, TVL=$64k)
            (ChainId.arbitrum, "0x4c4f752fa54dafb6d51b4a39018271c90ba1156f"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x4c4f752fa54dafb6d51b4a39018271c90ba1156f

            # [USDC] Yield Chasing USDC (protocol=goat-protocol, age=1.2y, CAGR 1m=3.78%, CAGR 3m=5.05%, CAGR 1y=7.31%, CAGR all=7.56%, TVL=$111k)
            (ChainId.arbitrum, "0x0df2e3a0b5997adc69f8768e495fd98a4d00f134"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x0df2e3a0b5997adc69f8768e495fd98a4d00f134

            # [USDC.e] Yield Chasing Silo USDC (protocol=goat-protocol, age=1.3y, CAGR 1m=2.20%, CAGR 3m=3.87%, CAGR 1y=7.15%, CAGR all=8.73%, TVL=$610k)
            (ChainId.arbitrum, "0x8a1ef3066553275829d1c0f64ee8d5871d5ce9d3"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x8a1ef3066553275829d1c0f64ee8d5871d5ce9d3

            # [USD₮0] Dolomite: USDT (protocol=dolomite, age=1.2y, CAGR 1m=3.78%, CAGR 3m=3.97%, CAGR 1y=6.88%, CAGR all=7.88%, TVL=$522k)
            (ChainId.arbitrum, "0xf2d2d55daf93b0660297eaa10969ebe90ead5ce8"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xf2d2d55daf93b0660297eaa10969ebe90ead5ce8

            # [USD₮0] DAMM Stablecoin Fund (protocol=lagoon-finance, age=0.6y, CAGR 1m=11.15%, CAGR 3m=7.00%, CAGR 1y=6.85%, CAGR all=6.85%, TVL=$1.8M)
            (ChainId.arbitrum, "0xe5d6eb448ac5a762c1ebe8cd1692b9cd08025176"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xe5d6eb448ac5a762c1ebe8cd1692b9cd08025176

            # [USDC] Dolomite: USDC (protocol=dolomite, age=1.2y, CAGR 1m=3.11%, CAGR 3m=4.05%, CAGR 1y=6.73%, CAGR all=7.83%, TVL=$5.9M)
            (ChainId.arbitrum, "0x444868b6e8079ac2c55eea115250f92c2b2c4d14"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x444868b6e8079ac2c55eea115250f92c2b2c4d14

            # [USDC] Hyperithm USDC (protocol=morpho, age=0.5y, CAGR 1m=4.05%, CAGR 3m=5.30%, CAGR 1y=6.71%, CAGR all=6.71%, TVL=$3.1M)
            (ChainId.arbitrum, "0x4b6f1c9e5d470b97181786b26da0d0945a7cf027"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x4b6f1c9e5d470b97181786b26da0d0945a7cf027

            # [USDC] Clearstar High Yield USDC (protocol=morpho, age=0.4y, CAGR 1m=4.23%, CAGR 3m=5.34%, CAGR 1y=6.53%, CAGR all=6.53%, TVL=$1.8M)
            (ChainId.arbitrum, "0x64ca76e2525fc6ab2179300c15e343d73e42f958"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x64ca76e2525fc6ab2179300c15e343d73e42f958

            # [USDC] Clearstar USDC Reactor (protocol=morpho, age=0.5y, CAGR 1m=3.61%, CAGR 3m=4.78%, CAGR 1y=6.46%, CAGR all=6.46%, TVL=$182k)
            (ChainId.arbitrum, "0xa53cf822fe93002aeae16d395cd823ece161a6ac"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xa53cf822fe93002aeae16d395cd823ece161a6ac

            # [USDC] Yearn Degen USDC (protocol=morpho, age=0.5y, CAGR 1m=3.99%, CAGR 3m=5.00%, CAGR 1y=6.24%, CAGR all=6.24%, TVL=$1.1M)
            (ChainId.arbitrum, "0x36b69949d60d06eccc14de0ae63f4e00cc2cd8b9"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x36b69949d60d06eccc14de0ae63f4e00cc2cd8b9

            # [USDC] Harvest: USDC Vault (0xB01a) (protocol=harvest-finance, age=1.4y, CAGR 1m=2.79%, CAGR 3m=3.66%, CAGR 1y=6.05%, CAGR all=7.44%, TVL=$93k)
            (ChainId.arbitrum, "0xb01a958d8e9dba566c6d71f66ef566ccf5fac859"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xb01a958d8e9dba566c6d71f66ef566ccf5fac859

            # [USDC.e] USDC-2 yVault (protocol=yearn, age=1.7y, CAGR 1m=2.30%, CAGR 3m=4.62%, CAGR 1y=8.35%, CAGR all=8.04%, TVL=$130k)
            # Deposits disabled: Max deposit cap reached (maxDeposit=0)
            (ChainId.arbitrum, "0x9fa306b1f4a6a83fec98d8ebbabedff78c407f6b"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x9fa306b1f4a6a83fec98d8ebbabedff78c407f6b

            #
            # Avalanche
            #
            # 7 vaults, sorted by 1y CAGR
            # Filter: min TVL $50k, min age 0.3y, denomination in (USDC, USDC.e, crvUSD, USDS, USDT/USD₮0), exclude Blacklisted/Dangerous
            #

            # [USDC] Borrowable USDC Deposit, SiloId: 142 (protocol=silo-finance, age=0.5y, CAGR 1m=4.76%, CAGR 3m=8.54%, CAGR 1y=9.62%, CAGR all=9.62%, TVL=$3.2M)
            (ChainId.avalanche, "0x606fe9a70338e798a292ca22c1f28c829f24048e"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x606fe9a70338e798a292ca22c1f28c829f24048e

            # [USDC] 9Summits USDC (protocol=euler, age=0.4y, CAGR 1m=6.47%, CAGR 3m=8.16%, CAGR 1y=7.79%, CAGR all=7.79%, TVL=$4.3M)
            (ChainId.avalanche, "0x4af3abe954259fb70b97c57ebd7ac1eb822028ef"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x4af3abe954259fb70b97c57ebd7ac1eb822028ef

            # [USDC] 9Summits USDC (protocol=euler, age=0.4y, CAGR 1m=6.40%, CAGR 3m=8.16%, CAGR 1y=7.78%, CAGR all=7.78%, TVL=$5.6M)
            (ChainId.avalanche, "0x37ca03ad51b8ff79aad35fadacba4cedf0c3e74e"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x37ca03ad51b8ff79aad35fadacba4cedf0c3e74e

            # [USDC] Re7 Labs USDC (protocol=euler, age=0.9y, CAGR 1m=0.18%, CAGR 3m=0.18%, CAGR 1y=5.46%, CAGR all=5.46%, TVL=$7.8M)
            (ChainId.avalanche, "0x39de0f00189306062d79edec6dca5bb6bfd108f9"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x39de0f00189306062d79edec6dca5bb6bfd108f9

            # [USDC] Re7 USDC (protocol=euler, age=0.5y, CAGR 1m=0.18%, CAGR 3m=0.18%, CAGR 1y=3.17%, CAGR all=3.17%, TVL=$2.9M)
            (ChainId.avalanche, "0xeaf77df5d03306bca4ee8b58b6821e6aca76309d"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xeaf77df5d03306bca4ee8b58b6821e6aca76309d

            # [USDt] K3 Capital USDT (protocol=term-finance, age=0.7y, CAGR 1m=0.00%, CAGR 3m=0.00%, CAGR 1y=3.13%, CAGR all=3.13%, TVL=$1.3M)
            (ChainId.avalanche, "0x8fc260cd0a00cac30eb1f444b8f1511d71420af9"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x8fc260cd0a00cac30eb1f444b8f1511d71420af9

            # [USDC] Turtle USDC (protocol=euler, age=0.4y, CAGR 1m=0.00%, CAGR 3m=0.00%, CAGR 1y=0.00%, CAGR all=0.00%, TVL=$61k)
            (ChainId.avalanche, "0xf8c100ab4b2d416609b0ef18105ebf601b2ec84a"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xf8c100ab4b2d416609b0ef18105ebf601b2ec84a

            # [USDC] Keyring USDC (protocol=euler, age=0.6y, CAGR 1m=0.00%, CAGR 3m=0.00%, CAGR 1y=4.35%, CAGR all=4.35%, TVL=$6.2M)
            # Deposits disabled: Max deposit cap reached (maxDeposit=0)
            (ChainId.avalanche, "0x8f23da78e3f31ab5deb75dc3282198bed630ffde"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x8f23da78e3f31ab5deb75dc3282198bed630ffde

            # [USDt] USDT Meta Vault (protocol=yearn, age=0.7y, CAGR 1m=0.00%, CAGR 3m=0.00%, CAGR 1y=2.98%, CAGR all=2.98%, TVL=$1.3M)
            # Deposits disabled: Max deposit cap reached (maxDeposit=0)
            (ChainId.avalanche, "0x39288474bc5931d3c4705e866b6e21cc2e47617d"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x39288474bc5931d3c4705e866b6e21cc2e47617d

            #
            # Hypercore
            #
            # 17 vaults, sorted by 1y CAGR
            # Filter: min TVL $50k, min age 0.3y, denomination in (USDC, USDC.e, crvUSD, USDS, USDT/USD₮0), exclude Blacklisted/Dangerous
            #

            # [USDC] pmalt (protocol=hyperliquid, age=1.4y, CAGR 1m=1036.67%, CAGR 3m=187.18%, CAGR 1y=1129.86%, CAGR all=395.64%, TVL=$696k)
            (HYPERCORE_CHAIN_ID, "0x4dec0a851849056e259128464ef28ce78afa27f6"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x4dec0a851849056e259128464ef28ce78afa27f6

            # [USDC] Loop Fund (protocol=hyperliquid, age=0.8y, CAGR 1m=797.98%, CAGR 3m=31.71%, CAGR 1y=593.33%, CAGR all=593.33%, TVL=$66k)
            (HYPERCORE_CHAIN_ID, "0xfeab64de8cdf9dcebc0f49812499e396273efc06"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xfeab64de8cdf9dcebc0f49812499e396273efc06

            # [USDC] Archangel Quant Fund I (protocol=hyperliquid, age=0.5y, CAGR 1m=141.25%, CAGR 3m=175.57%, CAGR 1y=194.59%, CAGR all=194.59%, TVL=$216k)
            (HYPERCORE_CHAIN_ID, "0x8c7bd04cf8d00d68ce8bc7d2f3f02f98d16a5ab0"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x8c7bd04cf8d00d68ce8bc7d2f3f02f98d16a5ab0

            # [USDC] OnlyShorts (protocol=hyperliquid, age=0.4y, CAGR 1m=11.70%, CAGR 3m=64.55%, CAGR 1y=154.73%, CAGR all=154.73%, TVL=$902k)
            (HYPERCORE_CHAIN_ID, "0x61b1cf5c2d7c4bf6d5db14f36651b2242e7cba0a"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x61b1cf5c2d7c4bf6d5db14f36651b2242e7cba0a

            # [USDC] Goon Edging (protocol=hyperliquid, age=1.1y, CAGR 1m=-90.19%, CAGR 3m=66.27%, CAGR 1y=136.49%, CAGR all=94.74%, TVL=$69k)
            (HYPERCORE_CHAIN_ID, "0x2431edfcb662e6ff6deab113cc91878a0b53fb0f"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x2431edfcb662e6ff6deab113cc91878a0b53fb0f

            # [USDC] Citadel (protocol=hyperliquid, age=2.3y, CAGR 1m=134.43%, CAGR 3m=158.38%, CAGR 1y=119.88%, CAGR all=417.51%, TVL=$602k)
            (HYPERCORE_CHAIN_ID, "0xda51323fe9800c8365646ad5c7ade0dd17fdc167"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xda51323fe9800c8365646ad5c7ade0dd17fdc167

            # [USDC] [ Systemic Strategies ] L/S Grids (protocol=hyperliquid, age=1.1y, CAGR 1m=-29.44%, CAGR 3m=3.94%, CAGR 1y=107.99%, CAGR all=58.97%, TVL=$3.3M)
            (HYPERCORE_CHAIN_ID, "0x07fd993f0fa3a185f7207adccd29f7a87404689d"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x07fd993f0fa3a185f7207adccd29f7a87404689d

            # [USDC] JizzJazz (protocol=hyperliquid, age=1.9y, CAGR 1m=15.22%, CAGR 3m=26.86%, CAGR 1y=77.76%, CAGR all=74.55%, TVL=$144k)
            (HYPERCORE_CHAIN_ID, "0x82eba5dc675279cb5967952f0c4b5184505eb17c"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x82eba5dc675279cb5967952f0c4b5184505eb17c

            # [USDC] Growi HF (protocol=hyperliquid, age=1.7y, CAGR 1m=59.56%, CAGR 3m=47.96%, CAGR 1y=53.02%, CAGR all=69.50%, TVL=$7.4M)
            (HYPERCORE_CHAIN_ID, "0x1e37a337ed460039d1b15bd3bc489de789768d5e"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x1e37a337ed460039d1b15bd3bc489de789768d5e

            # [USDC] 69 Jump Street (protocol=hyperliquid, age=1.1y, CAGR 1m=43.69%, CAGR 3m=22.59%, CAGR 1y=47.61%, CAGR all=58.53%, TVL=$527k)
            (HYPERCORE_CHAIN_ID, "0xa844d7ac9fa3424c4fd38a25baa23e460ec3e802"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xa844d7ac9fa3424c4fd38a25baa23e460ec3e802

            # [USDC] Aquila Chrysaetos (protocol=hyperliquid, age=0.6y, CAGR 1m=-11.30%, CAGR 3m=20.57%, CAGR 1y=37.85%, CAGR all=37.85%, TVL=$73k)
            (HYPERCORE_CHAIN_ID, "0x50e2fe552727a4b8692c192b4f96d1a6b0d44394"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x50e2fe552727a4b8692c192b4f96d1a6b0d44394

            # [USDC] HyperNeutral #1 (protocol=hyperliquid, age=0.7y, CAGR 1m=-0.76%, CAGR 3m=17.18%, CAGR 1y=29.90%, CAGR all=29.90%, TVL=$79k)
            (HYPERCORE_CHAIN_ID, "0x8bb033130be354eed1110ce7228d7095f001d9fe"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x8bb033130be354eed1110ce7228d7095f001d9fe

            # [USDC] [ Tachyon ] HYPE  (protocol=hyperliquid, age=0.3y, CAGR 1m=209.78%, CAGR 3m=41.55%, CAGR 1y=29.79%, CAGR all=29.79%, TVL=$56k)
            (HYPERCORE_CHAIN_ID, "0x131ab0c5032079bb9286ffc1828e11d5931e77bb"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x131ab0c5032079bb9286ffc1828e11d5931e77bb

            # [USDC] Crypto Trading Channel (protocol=hyperliquid, age=1.3y, CAGR 1m=7.95%, CAGR 3m=103.77%, CAGR 1y=24.41%, CAGR all=24.77%, TVL=$86k)
            (HYPERCORE_CHAIN_ID, "0x51b62b4bf8df6f2795b3da30cb46aa47f9f230a8"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x51b62b4bf8df6f2795b3da30cb46aa47f9f230a8

            # [USDC] Jay Pennies to Jay Dollars (protocol=hyperliquid, age=0.9y, CAGR 1m=3.32%, CAGR 3m=4.24%, CAGR 1y=21.80%, CAGR all=21.80%, TVL=$176k)
            (HYPERCORE_CHAIN_ID, "0x3b4f22366857da94f9346f88eac84718c8a8d48d"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x3b4f22366857da94f9346f88eac84718c8a8d48d

            # [USDC] AceVault Hyper01 (protocol=hyperliquid, age=0.6y, CAGR 1m=18.09%, CAGR 3m=9.12%, CAGR 1y=20.00%, CAGR all=20.00%, TVL=$326k)
            (HYPERCORE_CHAIN_ID, "0x9e02aca9865e1859bb7865f6f64801e804a173df"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x9e02aca9865e1859bb7865f6f64801e804a173df

            # [USDC] LowRiskCryptoGainer (protocol=hyperliquid, age=1.6y, CAGR 1m=24.43%, CAGR 3m=1.13%, CAGR 1y=19.75%, CAGR all=33.60%, TVL=$144k)
            (HYPERCORE_CHAIN_ID, "0xd57c9295947b5a616a3933344ef03a1ad67318ea"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xd57c9295947b5a616a3933344ef03a1ad67318ea

            # # [USDC] Sentiment Edge (protocol=hyperliquid, age=1.1y, CAGR 1m=-27.36%, CAGR 3m=-50.79%, CAGR 1y=72.66%, CAGR all=99.97%, TVL=$110k)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0x026a2e082a03200a00a97974b7bf7753ce33540f"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0x026a2e082a03200a00a97974b7bf7753ce33540f

            # # [USDC] HyperTwin - Growi HF 2x (protocol=hyperliquid, age=0.7y, CAGR 1m=554.88%, CAGR 3m=52.13%, CAGR 1y=8.63%, CAGR all=8.63%, TVL=$131k)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0x15be61aef0ea4e4dc93c79b668f26b3f1be75a66"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0x15be61aef0ea4e4dc93c79b668f26b3f1be75a66

            # # [USDC] MOAS (protocol=hyperliquid, age=1.3y, CAGR 1m=211.99%, CAGR 3m=885.11%, CAGR 1y=19.55%, CAGR all=0.40%, TVL=$139k)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0x29b98aaf8eeb316385fe2ed1af564bdc4b03ffd6"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0x29b98aaf8eeb316385fe2ed1af564bdc4b03ffd6

            # # [USDC] Cryptoaddcited (protocol=hyperliquid, age=1.1y, CAGR 1m=-2.02%, CAGR 3m=42.57%, CAGR 1y=25.87%, CAGR all=40.17%, TVL=$116k)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0x5108cd0a328ed28c277f958761fe1cda60c21aa8"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0x5108cd0a328ed28c277f958761fe1cda60c21aa8

            # # [USDC] +convexity (protocol=hyperliquid, age=0.5y, CAGR 1m=-16.86%, CAGR 3m=33.64%, CAGR 1y=27.35%, CAGR all=27.35%, TVL=$490k)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0x5661a070eb13c7c55ac3210b2447d4bea426cbf5"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0x5661a070eb13c7c55ac3210b2447d4bea426cbf5

            # # [USDC] BULBUL2DAO (protocol=hyperliquid, age=1.2y, CAGR 1m=452.62%, CAGR 3m=132.45%, CAGR 1y=177.51%, CAGR all=157.29%, TVL=$97k)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0x65aee08c9235025355ac6c5ad020fb167ecef4fe"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0x65aee08c9235025355ac6c5ad020fb167ecef4fe

            # # [USDC] ski lambo beach  (protocol=hyperliquid, age=1.1y, CAGR 1m=-99.84%, CAGR 3m=-87.42%, CAGR 1y=173.85%, CAGR all=114.76%, TVL=$78k)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0x66e541024ca4c50b8f6c0934b8947c487d211661"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0x66e541024ca4c50b8f6c0934b8947c487d211661

            # # [USDC] Long LINK Short XRP (protocol=hyperliquid, age=0.6y, CAGR 1m=-25.10%, CAGR 3m=-99.83%, CAGR 1y=-88.31%, CAGR all=-88.31%, TVL=$835k)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0x73ce82fb75868af2a687e9889fcf058dd1cf8ce9"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0x73ce82fb75868af2a687e9889fcf058dd1cf8ce9

            # # [USDC] Long HYPE & BTC | Short Garbage (protocol=hyperliquid, age=0.9y, CAGR 1m=71.18%, CAGR 3m=2302.55%, CAGR 1y=181.04%, CAGR all=181.04%, TVL=$1.8M)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0xac26cf5f3c46b5e102048c65b977d2551b72a9c7"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0xac26cf5f3c46b5e102048c65b977d2551b72a9c7

            # # [USDC] Crypto_Lab28 (protocol=hyperliquid, age=0.7y, CAGR 1m=45.67%, CAGR 3m=127.65%, CAGR 1y=66.59%, CAGR all=66.59%, TVL=$155k)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0xb11fe7f2e97bd02b2da909b32f4a5e7fcb0df099"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0xb11fe7f2e97bd02b2da909b32f4a5e7fcb0df099

            # # [USDC] Sentiment Edge  (protocol=hyperliquid, age=1.0y, CAGR 1m=-23.00%, CAGR 3m=-34.75%, CAGR 1y=84.60%, CAGR all=87.87%, TVL=$166k)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0xb7e7d0fdeff5473ed6ef8d3a762d096a040dbb18"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0xb7e7d0fdeff5473ed6ef8d3a762d096a040dbb18

            # # [USDC] Jade Lotus Capital (protocol=hyperliquid, age=0.8y, CAGR 1m=-87.87%, CAGR 3m=-68.63%, CAGR 1y=402.07%, CAGR all=402.07%, TVL=$327k)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0xbc5bf88fd012612ba92c5bd96e183955801b7fdc"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0xbc5bf88fd012612ba92c5bd96e183955801b7fdc

            # # [USDC] BTC/ETH CTA | AIM (protocol=hyperliquid, age=0.7y, CAGR 1m=-65.12%, CAGR 3m=-3.72%, CAGR 1y=13.97%, CAGR all=13.97%, TVL=$96k)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0xbeebbbe817a69d60dd62e0a942032bc5414dae1c"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0xbeebbbe817a69d60dd62e0a942032bc5414dae1c

            # # [USDC] hidden marko fund (protocol=hyperliquid, age=0.7y, CAGR 1m=-89.63%, CAGR 3m=-67.38%, CAGR 1y=-24.24%, CAGR all=-24.24%, TVL=$120k)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0xc497f1f8840dd65affbab1a610b6e558844743d4"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0xc497f1f8840dd65affbab1a610b6e558844743d4

            # # [USDC] Sifu (protocol=hyperliquid, age=2.2y, CAGR 1m=-99.87%, CAGR 3m=25.02%, CAGR 1y=-96.72%, CAGR all=-99.73%, TVL=$1.4M)
            # # Excluded vault
            # (HYPERCORE_CHAIN_ID, "0xf967239debef10dbc78e9bbbb2d8a16b72a614eb"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0xf967239debef10dbc78e9bbbb2d8a16b72a614eb

            #
            # HyperEVM
            #
            # 15 vaults, sorted by 1y CAGR
            # Filter: min TVL $50k, min age 0.3y, denomination in (USDC, USDC.e, crvUSD, USDS, USDT/USD₮0), exclude Blacklisted/Dangerous
            #

            # [USD₮0] hyUSD₮0 (Looped HYPE) - 9 (protocol=hypurrfi, age=0.7y, CAGR 1m=6.09%, CAGR 3m=8.14%, CAGR 1y=11.71%, CAGR all=11.71%, TVL=$172k)
            (ChainId.hyperliquid, "0x1c5164a764844356d57654ea83f9f1b72cd10db5"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x1c5164a764844356d57654ea83f9f1b72cd10db5

            # [USD₮0] HYPE++ Hyperliquid (protocol=d2-finance, age=0.6y, CAGR 1m=18.99%, CAGR 3m=8.75%, CAGR 1y=10.95%, CAGR all=10.95%, TVL=$988k)
            (ChainId.hyperliquid, "0x195eb4d088f222c982282b5dd495e76dba4bc7d1"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x195eb4d088f222c982282b5dd495e76dba4bc7d1

            # [USD₮0] hyUSD₮0 (hwHLP) - 11 (protocol=hypurrfi, age=0.7y, CAGR 1m=9.05%, CAGR 3m=9.12%, CAGR 1y=9.86%, CAGR all=9.86%, TVL=$159k)
            (ChainId.hyperliquid, "0x2c910f67dbf81099e6f8e126e7265d7595dc20ad"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x2c910f67dbf81099e6f8e126e7265d7595dc20ad

            # [USD₮0] MEV Capital USDT0 (protocol=morpho, age=0.6y, CAGR 1m=6.53%, CAGR 3m=6.77%, CAGR 1y=8.35%, CAGR all=8.35%, TVL=$51k)
            (ChainId.hyperliquid, "0x3bcc0a5a66bb5bdceef5dd8a659a4ec75f3834d8"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x3bcc0a5a66bb5bdceef5dd8a659a4ec75f3834d8

            # [USD₮0] Hyperithm USDT0 (protocol=morpho, age=0.6y, CAGR 1m=6.96%, CAGR 3m=5.69%, CAGR 1y=8.25%, CAGR all=8.25%, TVL=$468k)
            (ChainId.hyperliquid, "0xe5add96840f0b908ddeb3bd144c0283ac5ca7ca0"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xe5add96840f0b908ddeb3bd144c0283ac5ca7ca0

            # [USD₮0] Felix USDT0 (Frontier) (protocol=morpho, age=0.6y, CAGR 1m=5.79%, CAGR 3m=4.59%, CAGR 1y=8.11%, CAGR all=8.11%, TVL=$2.5M)
            (ChainId.hyperliquid, "0x9896a8605763106e57a51aa0a97fe8099e806bb3"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x9896a8605763106e57a51aa0a97fe8099e806bb3

            # [USDC] Gauntlet USDC (protocol=morpho, age=0.4y, CAGR 1m=4.83%, CAGR 3m=5.10%, CAGR 1y=7.32%, CAGR all=7.32%, TVL=$2.0M)
            (ChainId.hyperliquid, "0x08c00f8279dff5b0cb5a04d349e7d79708ceadf3"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x08c00f8279dff5b0cb5a04d349e7d79708ceadf3

            # [USD₮0] Felix USDT0 (protocol=morpho, age=0.8y, CAGR 1m=5.27%, CAGR 3m=3.79%, CAGR 1y=7.21%, CAGR all=7.21%, TVL=$12.4M)
            (ChainId.hyperliquid, "0xfc5126377f0efc0041c0969ef9ba903ce67d151e"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xfc5126377f0efc0041c0969ef9ba903ce67d151e

            # [USDC] Felix USDC (protocol=morpho, age=0.5y, CAGR 1m=4.73%, CAGR 3m=4.86%, CAGR 1y=6.71%, CAGR all=6.71%, TVL=$29.8M)
            (ChainId.hyperliquid, "0x8a862fd6c12f9ad34c9c2ff45ab2b6712e8cea27"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x8a862fd6c12f9ad34c9c2ff45ab2b6712e8cea27

            # [USD₮0] Gauntlet USDT0 Vault (protocol=morpho, age=0.8y, CAGR 1m=6.39%, CAGR 3m=4.45%, CAGR 1y=6.49%, CAGR all=6.49%, TVL=$369k)
            (ChainId.hyperliquid, "0x53a333e51e96fe288bc9add7cdc4b1ead2cd2ffa"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x53a333e51e96fe288bc9add7cdc4b1ead2cd2ffa

            # [USD₮0] Static USD₮0 Hypurr (protocol=superform, age=0.5y, CAGR 1m=3.93%, CAGR 3m=3.27%, CAGR 1y=5.88%, CAGR all=5.88%, TVL=$152k)
            (ChainId.hyperliquid, "0xdc6f4239c1d8d3b955c06cb8f1a6cf18effc5bfe"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xdc6f4239c1d8d3b955c06cb8f1a6cf18effc5bfe

            # [USD₮0] Hyperdrive Treasury USD₮0 Market (protocol=unknown, age=0.6y, CAGR 1m=4.71%, CAGR 3m=3.68%, CAGR 1y=5.21%, CAGR all=5.21%, TVL=$62k)
            (ChainId.hyperliquid, "0x260f5f56ad7d14789d43fd538429d42ff5b82b56"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x260f5f56ad7d14789d43fd538429d42ff5b82b56

            # [USD₮0] Hyperdrive Primary USD₮0 Market (protocol=unknown, age=0.8y, CAGR 1m=0.59%, CAGR 3m=2.10%, CAGR 1y=4.95%, CAGR all=4.95%, TVL=$545k)
            (ChainId.hyperliquid, "0xa522572cf63e6ceed49db6c77cd9ec76c1d47d09"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xa522572cf63e6ceed49db6c77cd9ec76c1d47d09

            # [USDC] MEV Capital USDC (protocol=morpho, age=0.4y, CAGR 1m=4.91%, CAGR 3m=2.43%, CAGR 1y=1.89%, CAGR all=1.89%, TVL=$67k)
            (ChainId.hyperliquid, "0x4851d4891321035729713d43be1f4bb883dffd34"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x4851d4891321035729713d43be1f4bb883dffd34

            # [USDC] Hyperdrive Primary USDC Market (protocol=unknown, age=0.5y, CAGR 1m=0.20%, CAGR 3m=0.59%, CAGR 1y=1.75%, CAGR all=1.75%, TVL=$72k)
            (ChainId.hyperliquid, "0xeb3f65fc7011fb836a8cb6a136c9c5099fda2837"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xeb3f65fc7011fb836a8cb6a136c9c5099fda2837

            # # [USD₮0] Wrapped HLP (protocol=hyperlend, age=0.7y, CAGR 1m=8.50%, CAGR 3m=9.97%, CAGR 1y=12.14%, CAGR all=12.14%, TVL=$233k)
            # # Excluded vault
            # (ChainId.hyperliquid, "0x06fd9d03b3d0f18e4919919b72d30c582f0a97e5"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0x06fd9d03b3d0f18e4919919b72d30c582f0a97e5

            #
            # Monad
            #
            # 14 vaults, sorted by 1y CAGR
            # Filter: min TVL $50k, min age 0.1y, denomination in (USDC, USDC.e, crvUSD, USDS, USDT/USD₮0), exclude Blacklisted/Dangerous
            #

            # [USDC] Hyperithm USDC Degen (protocol=morpho, age=0.2y, CAGR 1m=4.43%, CAGR 3m=4.30%, CAGR 1y=4.30%, CAGR all=4.30%, TVL=$11.0M)
            (ChainId.monad, "0xa8665084d8cd6276c00ca97cbc0bf4bc9ae94c79"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xa8665084d8cd6276c00ca97cbc0bf4bc9ae94c79

            # [USDC] Curvance USDC (protocol=curvance, age=0.3y, CAGR 1m=3.78%, CAGR 3m=4.29%, CAGR 1y=3.75%, CAGR all=3.75%, TVL=$1.3M)
            (ChainId.monad, "0x8ee9fc28b8da872c38a496e9ddb9700bb7261774"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x8ee9fc28b8da872c38a496e9ddb9700bb7261774

            # [USDC] MuDigital Tulipa USDC (protocol=lagoon-finance, age=0.3y, CAGR 1m=2.49%, CAGR 3m=3.57%, CAGR 1y=3.57%, CAGR all=3.57%, TVL=$3.0M)
            (ChainId.monad, "0x0da39b740834090c146dc48357f6a435a1bb33b3"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x0da39b740834090c146dc48357f6a435a1bb33b3

            # [USDC] Steakhouse High Yield USDC (protocol=morpho, age=0.3y, CAGR 1m=3.91%, CAGR 3m=3.83%, CAGR 1y=3.22%, CAGR all=3.22%, TVL=$7.2M)
            (ChainId.monad, "0x802c91d807a8daca257c4708ab264b6520964e44"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x802c91d807a8daca257c4708ab264b6520964e44

            # [USDC] EDGE UltraYield USDC (protocol=gearbox, age=0.3y, CAGR 1m=1.67%, CAGR 3m=1.76%, CAGR 1y=1.49%, CAGR all=1.49%, TVL=$15.2M)
            (ChainId.monad, "0x6b343f7b797f1488aa48c49d540690f2b2c89751"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x6b343f7b797f1488aa48c49d540690f2b2c89751

            # [USDT0] Steakhouse High Yield USDT0 (protocol=morpho, age=0.3y, CAGR 1m=N/A, CAGR 3m=0.79%, CAGR 1y=0.86%, CAGR all=0.86%, TVL=$3.5M)
            (ChainId.monad, "0x961a59fe249b9795fae7fa35f9e89629689d5278"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x961a59fe249b9795fae7fa35f9e89629689d5278

            # [USDC] Isolated WBTC-USDC USDC (protocol=euler, age=0.3y, CAGR 1m=0.93%, CAGR 3m=0.90%, CAGR 1y=0.77%, CAGR all=0.77%, TVL=$1.4M)
            (ChainId.monad, "0xf19e8ddc541dee2f4d6796a79b1c1e10a415a0da"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xf19e8ddc541dee2f4d6796a79b1c1e10a415a0da

            # [USDC] Isolated wstETH-WETH-USDC USDC (protocol=euler, age=0.3y, CAGR 1m=0.83%, CAGR 3m=0.88%, CAGR 1y=0.76%, CAGR all=0.76%, TVL=$2.0M)
            (ChainId.monad, "0x1e4d67c666c2ccf27a0af980fe6c8e0f05ac8949"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x1e4d67c666c2ccf27a0af980fe6c8e0f05ac8949

            # [USDC] Euler Earn USDC (protocol=euler, age=0.3y, CAGR 1m=0.43%, CAGR 3m=0.45%, CAGR 1y=0.39%, CAGR all=0.39%, TVL=$6.9M)
            (ChainId.monad, "0xa981f053c118fe4db0e1aeba192aad20ec7f7801"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xa981f053c118fe4db0e1aeba192aad20ec7f7801

            # [USDC] Isolated XAUT-USDC USDC (protocol=euler, age=0.3y, CAGR 1m=0.23%, CAGR 3m=0.14%, CAGR 1y=0.13%, CAGR all=0.13%, TVL=$53k)
            (ChainId.monad, "0x289f801765b99b5e6263853859fe302dbecab6d6"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x289f801765b99b5e6263853859fe302dbecab6d6

            # [USDC] Isolated shMON-WMON-USDC USDC (protocol=euler, age=0.3y, CAGR 1m=0.19%, CAGR 3m=0.14%, CAGR 1y=0.12%, CAGR all=0.12%, TVL=$2.0M)
            (ChainId.monad, "0x5792753b66eb5213e416755546abbcc1aef1008a"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x5792753b66eb5213e416755546abbcc1aef1008a

            # [USDC] Isolated gMON-WMON-USDC USDC (protocol=euler, age=0.3y, CAGR 1m=0.16%, CAGR 3m=0.11%, CAGR 1y=0.10%, CAGR all=0.10%, TVL=$393k)
            (ChainId.monad, "0xf2acb868eaf64ef36bc0d3b4ba93af00fc3167cf"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xf2acb868eaf64ef36bc0d3b4ba93af00fc3167cf

            # [USDC] Isolated LBTC-USDC USDC (protocol=euler, age=0.3y, CAGR 1m=0.01%, CAGR 3m=0.01%, CAGR 1y=0.01%, CAGR all=0.01%, TVL=$395k)
            (ChainId.monad, "0xf62c7cd4f71434f7c4d12a1142133d4ed4abaec4"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xf62c7cd4f71434f7c4d12a1142133d4ed4abaec4

            # [USDC] Isolated sMON-WMON-USDC USDC (protocol=euler, age=0.3y, CAGR 1m=0.01%, CAGR 3m=0.00%, CAGR 1y=0.00%, CAGR all=0.00%, TVL=$373k)
            (ChainId.monad, "0x36d0f0a4cb615c200310ae2c5b5d345189d01197"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x36d0f0a4cb615c200310ae2c5b5d345189d01197

            # [USDC] Hyperithm USDC Degen (protocol=morpho, age=0.2y, CAGR 1m=4.43%, CAGR 3m=4.30%, CAGR 1y=4.30%, CAGR all=4.30%, TVL=$11.0M)
            # Deposits disabled: Max deposit cap reached (maxDeposit=0)
            (ChainId.monad, "0x78999cc96d2ba0341588c60ccb0e91c6c33cf371"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0x78999cc96d2ba0341588c60ccb0e91c6c33cf371

            # [USDC] Steakhouse High Yield USDC (protocol=morpho, age=0.3y, CAGR 1m=3.91%, CAGR 3m=3.83%, CAGR 1y=3.31%, CAGR all=3.31%, TVL=$7.2M)
            # Deposits disabled: Max deposit cap reached (maxDeposit=0)
            (ChainId.monad, "0xbeeff443c3cba3e369da795002243beac311ab83"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xbeeff443c3cba3e369da795002243beac311ab83

            # [USDT0] Steakhouse High Yield USDT0 (protocol=morpho, age=0.3y, CAGR 1m=N/A, CAGR 3m=0.79%, CAGR 1y=0.87%, CAGR all=0.87%, TVL=$3.5M)
            # Deposits disabled: Max deposit cap reached (maxDeposit=0)
            (ChainId.monad, "0xbeeff300e9a9caec7beea740ab8758d33b777509"),
            # https://tradingstrategy.ai/trading-view/vaults/address/0xbeeff300e9a9caec7beea740ab8758d33b777509

            # # [USDC] Aegis Yield Vault (protocol=accountable, age=0.2y, CAGR 1m=2.81%, CAGR 3m=3.14%, CAGR 1y=3.14%, CAGR all=3.14%, TVL=$220k)
            # # Excluded protocol (accountable): Assets are illiquid for strategies
            # (ChainId.monad, "0x0a4afb907672279926c73dc1f77151931c2a55cc"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0x0a4afb907672279926c73dc1f77151931c2a55cc

            # # [USDC] Yuzu Money Vault (protocol=accountable, age=0.2y, CAGR 1m=7.17%, CAGR 3m=8.73%, CAGR 1y=8.73%, CAGR all=8.73%, TVL=$14.9M)
            # # Excluded protocol (accountable): Assets are illiquid for strategies
            # (ChainId.monad, "0x3a2c4aaae6776dc1c31316de559598f2f952e2cb"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0x3a2c4aaae6776dc1c31316de559598f2f952e2cb

            # # [USDC] sUSN Delta Neutral Yield Vault (protocol=accountable, age=0.3y, CAGR 1m=7.91%, CAGR 3m=8.54%, CAGR 1y=8.37%, CAGR all=8.37%, TVL=$4.0M)
            # # Excluded protocol (accountable): Assets are illiquid for strategies
            # (ChainId.monad, "0x58ba69b289de313e66a13b7d1f822fc98b970554"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0x58ba69b289de313e66a13b7d1f822fc98b970554

            # # [USDC] Hyperithm Delta Neutral Vault (protocol=accountable, age=0.2y, CAGR 1m=10.05%, CAGR 3m=8.62%, CAGR 1y=8.62%, CAGR all=8.62%, TVL=$25.2M)
            # # Excluded protocol (accountable): Assets are illiquid for strategies
            # (ChainId.monad, "0x7cd231120a60f500887444a9baf5e1bd753a5e59"),
            # # https://tradingstrategy.ai/trading-view/vaults/address/0x7cd231120a60f500887444a9baf5e1bd753a5e59
        ]

        # Exclude Euro vaults, etc.
        # ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC", "USDT", "USDC.e"}
        ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC", "USDT", "USDC.e", "crvUSD", "USD₮0", "USDt", "USDT0", "USDS"}

        BENCHMARK_PAIRS = [
            (ChainId.ethereum, "uniswap-v3", "WBTC", "USDC", 0.003),
        ]
    case ChainId.arbitrum:

        EXCHANGES = ("uniswap-v2", "uniswap-v3")
        SUPPORTING_PAIRS = [
            (ChainId.arbitrum, "uniswap-v3", "WETH", "USDC", 0.0005),
        ]
        LENDING_RESERVES = None
        PREFERRED_STABLECOIN = USDC_NATIVE_TOKEN[PRIMARY_CHAIN_ID].lower()

        VAULTS = [
            # Arbitrum only
            (ChainId.arbitrum, "0x58bfc95a864e18e8f3041d2fcd3418f48393fe6a"),  # Plutus Hedge Token
            (ChainId.arbitrum, "0x959f3807f0aa7921e18c78b00b2819ba91e52fef"),  # gmUSDC
            (ChainId.arbitrum, "0xd3443ee1e91af28e5fb858fbd0d72a63ba8046e0"),  # gTrade (Gains) USDC
            (ChainId.arbitrum, "0x75288264fdfea8ce68e6d852696ab1ce2f3e5004"),  # Hype++
            (ChainId.arbitrum, "0x4b6f1c9e5d470b97181786b26da0d0945a7cf027"),  # Hypertrim USDC
            (ChainId.arbitrum, "0x0b2b2b2076d95dda7817e785989fe353fe955ef9"),  # Staked USDai
            (ChainId.arbitrum, "0x64ca76e2525fc6ab2179300c15e343d73e42f958"),  # Clearstar high yielsd USDC
            (ChainId.arbitrum, "0x7e97fa6893871a2751b5fe961978dccb2c201e65"),  # Gauntlet
            (ChainId.arbitrum, "0x1a996cb54bb95462040408c06122d45d6cdb6096"),  # Fluid
            (ChainId.arbitrum, "0xa91267a25939b2b0f046013fbf9597008f7f014b"),  # IPOR USDC Arbirum optimise
            (ChainId.arbitrum, "0x05d28a86e057364f6ad1a88944297e58fc6160b3"),  # Euler Arbitrum Yield USDC
            (ChainId.arbitrum, "0xc8248953429d707c6a2815653eca89846ffaa63b"),  # Curve LLAMMA asdCRV / crvUSD
            (ChainId.arbitrum, "0xf63b7f49b4f5dc5d0e7e583cfd79dc64e646320c"),  # Auto finance Tokemak ARB/USDC
            (ChainId.arbitrum, "0xeeaf2ccb73a01deb38eca2947d963d64cfde6a32"),  # Curve LLAMMA CRV / crvUSD
            (ChainId.arbitrum, "0xe5d6eb448ac5a762c1ebe8cd1692b9cd08025176"),  # DAMM stablecoin fund
        ]

        # Exclude Euro vaults, etc.
        # ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC", "USDT", "USDC.e"}
        ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC"}
    
    case ChainId.base:
        EXCHANGES = ("uniswap-v2", "uniswap-v3", "aerodrome")
        SUPPORTING_PAIRS = [
            (ChainId.base, "uniswap-v3", "WETH", "USDC", 0.0005),
        ]
        LENDING_RESERVES = None
        PREFERRED_STABLECOIN = USDC_NATIVE_TOKEN[ChainId.base].lower()

        VAULTS = [
            (ChainId.base, "0x13cd7cf42ccbaca8cd97e7f09572b6ea0de1097b"), # USDC-TIBBIR shares
        ]

        ALLOWED_VAULT_DENOMINATION_TOKENS = {"USDC"}


print(f"Chain universe: {CHAIN_ID}")
print(f"Vaults count: {len(VAULTS)}")
print(f"Allowed denomination tokens: {ALLOWED_VAULT_DENOMINATION_TOKENS}")

## Parameters

- Collection of parameters used in the calculations

In [ ]:
import datetime

import pandas as pd


from tradingstrategy.timebucket import TimeBucket
from tradeexecutor.strategy.cycle import CycleDuration
from tradeexecutor.strategy.parameters import StrategyParameters
from skopt.space import Categorical
from tradeexecutor.strategy.default_routing_options import TradeRouting

from tradeexecutor.utils.jupyter_notebook_name import get_notebook_id


class Parameters:

    id = get_notebook_id(globals())

    # We trade 1h candle
    candle_time_bucket = TimeBucket.d1
    cycle_duration = CycleDuration.cycle_4d
    
    chain_id = CHAIN_ID
    primary_chain_id = PRIMARY_CHAIN_ID
    exchanges = EXCHANGES
    
    #
    # Basket size, risk and balancing parametrs.
    #   
    min_asset_universe = 5  # How many assets we need in the asset universe to start running the index
    max_assets_in_portfolio = Categorical([5, 10, 20, 35])  # How many assets our basket can hold once
    allocation = 0.95  # Allocate all cash to volatile pairs
    # min_rebalance_trade_threshold_pct = 0.05  # % of portfolio composition must change before triggering rebalacne
    individual_rebalance_min_threshold_usd = 50.0  # Don't make buys less than this amount
    sell_rebalance_min_threshold = 10.0
    sell_threshold = 0.05  # Sell if asset is more than 5% of the portfolio
    per_position_cap_of_pool = 0.10  # Never own more than % of the lit liquidity of the trading pool
    max_concentration = Categorical([0.05, 0.10, 0.15, 0.20, 0.33]) # How large % can one asset be in a portfolio once
    min_portfolio_weight = 0.0050  # Close position / do not open if weight is less than 50 BPS

    # Because backtest does not have enough data at the early universe,
    # we tweak it to simulate taking more concentrated positions in the past to 
    # avoid unused cash
    early_allocation_vault_count_threshold = 20
    early_allocation_max_contration = 15

    # How long 
    # Needed to calculate weights
    rolling_returns_bars = Categorical([30, 60, 120, 180, 240])
    weighting_method = Categorical(["rolling_returns", "rolling_sharpe", "rolling_sortino", "inverse_volatility"])
    weight_function = Categorical(["weight_equal", "weight_1_slash_n"])
    volatility_window = 180  # 6 months lookback for annualised volatility
    min_tvl = 50_000  # Minimum TVL in the vault before it can be considered investable

    #     
    #
    # Backtesting only
    # Limiting factor: Aave v3 on Base starts at the end of DEC 2023
    #
    backtest_start = datetime.datetime(2024, 10, 1)
    backtest_end = datetime.datetime(2026, 3, 11)
    initial_cash = 100_000

    #
    # Live only
    #
    routing = TradeRouting.default
    required_history_period = datetime.timedelta(days=60*2)
    slippage_tolerance = 0.0060  # 0.6% 
    assummed_liquidity_when_data_missings = 10_000
    

parameters = StrategyParameters.from_class(Parameters)  # Convert to AttributedDict to easier typing with dot notation

from tradeexecutor.strategy.parameters import display_parameters
display_parameters(parameters)


# Trading pairs and market data

- This creates the strategy universe containing pair metadata and their prices
- The universe is "masked" by simply selecting pairs on the predefined pairs list

In [ ]:

from pathlib import Path
from typing import Callable
from tradingstrategy.pair import PandasPairUniverse


from eth_defi.vault.vaultdb import DEFAULT_VAULT_DATABASE, DEFAULT_RAW_PRICE_DATABASE


from tradingstrategy.utils.token_filter import filter_for_selected_pairs
from tradingstrategy.alternative_data.vault import load_vault_database


from tradeexecutor.strategy.trading_strategy_universe import TradingStrategyUniverse
from tradeexecutor.strategy.execution_context import ExecutionContext, notebook_execution_context
from tradeexecutor.strategy.universe_model import UniverseOptions
from tradeexecutor.strategy.trading_strategy_universe import TradingStrategyUniverse, load_partial_data
from tradeexecutor.strategy.execution_context import ExecutionContext, notebook_execution_context
from tradeexecutor.strategy.universe_model import UniverseOptions
from tradeexecutor.analysis.pair import display_strategy_universe
from tradeexecutor.strategy.pandas_trader.trading_universe_input import CreateTradingUniverseInput
from tradeexecutor.analysis.vault import display_vaults



# Hack to support vault data exposure to live trading universe creation
from dotenv import load_dotenv
load_dotenv(override=True)  # Loads variables from .env file


def create_trading_universe(
    input: CreateTradingUniverseInput,
) -> TradingStrategyUniverse:
    """Create the trading universe.

    - Load Trading Strategy full pairs dataset

    - Load built-in Coingecko top 1000 dataset

    - Get all DEX tokens for a certain Coigecko category

    - Load OHCLV data for these pairs

    - Load also BTC and ETH price data to be used as a benchmark
    """

    execution_context = input.execution_context
    client = input.client
    timestamp = input.timestamp
    parameters = input.parameters or Parameters  # Some CLI commands do not support yet passing this
    universe_options = input.universe_options

    if execution_context.live_trading:
        # Live trading, send strategy universe formation details
        # to logs
        debug_printer = logger.info
    else:
        # Jupyter notebook inline output
        debug_printer = print

    chain_id = parameters.primary_chain_id

    debug_printer(f"Preparing trading universe on chain {chain_id.get_name()}")

    # Pull out our benchmark pairs ids.
    # We need to construct pair universe object for the symbolic lookup.
    # TODO: PandasPairUniverse(buidl_index=True) - speed this up by skipping index building
    all_pairs_df = client.fetch_pair_universe().to_pandas()
    pairs_df= filter_for_selected_pairs(
        all_pairs_df,
        SUPPORTING_PAIRS,
    )    

    debug_printer(f"We have total {len(all_pairs_df)} pairs in dataset and going to use {len(pairs_df)} pairs for the strategy")

    # Check which vaults we can include based on allowed deposit tokens for this backtest
    # Use local vault database (~/.tradingstrategy/vaults/) instead of
    # the bundled one, as the bundled one may not have all chains
    vault_universe = load_vault_database(path=DEFAULT_VAULT_DATABASE)
    total_vaults = vault_universe.get_vault_count()
    vault_universe = vault_universe.limit_to_vaults(VAULTS, check_all_vaults_found=True)
    vault_universe = vault_universe.limit_to_denomination(ALLOWED_VAULT_DENOMINATION_TOKENS, check_all_vaults_found=True)
    debug_printer(f"Loaded total {vault_universe.get_vault_count()} vaults from the total of {total_vaults} in vault database, source vaults count: {len(VAULTS)}")

    # Default vault data bundle path for backtesting
    vault_bundled_price_data = DEFAULT_RAW_PRICE_DATABASE
    debug_printer(f"Using vault price data for backtesting from {vault_bundled_price_data}")

    dataset = load_partial_data(
        client=client,
        time_bucket=parameters.candle_time_bucket,
        pairs=pairs_df,
        execution_context=execution_context,
        universe_options=universe_options,
        liquidity=True,
        liquidity_time_bucket=TimeBucket.d1,
        lending_reserves=LENDING_RESERVES,
        vaults=vault_universe,
        vault_bundled_price_data=vault_bundled_price_data,
        check_all_vaults_found=True,
    )

    debug_printer("Creating strategy universe with price feeds and vaults")
    strategy_universe = TradingStrategyUniverse.create_from_dataset(
        dataset,
        reserve_asset=PREFERRED_STABLECOIN,
        forward_fill=True,  # We got very gappy data from low liquid DEX coins
        forward_fill_until=timestamp,
        primary_chain=parameters.primary_chain_id
    )
    return strategy_universe


universe_input = CreateTradingUniverseInput(
    execution_context=notebook_execution_context,
    client=client,
    timestamp=None,
    parameters=parameters,
    universe_options=UniverseOptions.from_strategy_parameters_class(Parameters, notebook_execution_context),
    execution_model=None,
)

strategy_universe = create_trading_universe(universe_input)


print("Universe creation done")

df = display_strategy_universe(
    strategy_universe, 
    sort_key="base",
    sort_numeric=False,
    limit=75,
    show_token_risk=True,
    show_tokensniffer=True,
)

df = df.head(3)

from tradeexecutor.utils.notebook import display_dataframe_with_html

display_dataframe_with_html(df)

# Indicators

- Precalculate indicators used by the strategy

In [ ]:
import pandas as pd
from IPython.display import HTML
import pandas_ta

from tradeexecutor.strategy.pandas_trader.indicator import IndicatorSource
from tradeexecutor.strategy.trading_strategy_universe import TradingStrategyUniverse
from tradeexecutor.strategy.pandas_trader.indicator import calculate_and_load_indicators_inline
from tradeexecutor.strategy.pandas_trader.indicator import IndicatorDependencyResolver
from tradeexecutor.state.types import USDollarAmount
from tradeexecutor.strategy.pandas_trader.indicator_decorator import IndicatorRegistry
from tradeexecutor.analysis.indicator import display_indicators
from tradeexecutor.state.identifier import TradingPairIdentifier


indicators = IndicatorRegistry()

empty_series = pd.Series([], index=pd.DatetimeIndex([]))


@indicators.define()
def rolling_returns(
    close: pd.Series,
    rolling_returns_bars: int = 60,
) -> pd.Series:
    """Calculate annualised rolling returns.

    - Use last 60 days of price data (max)
    - Require minimum 7 days of data
    - Annualise all returns to be comparable across different data lengths
    """
    min_periods = 7

    first_price = close.rolling(
        window=rolling_returns_bars,
        min_periods=min_periods,
    ).apply(lambda x: x[0], raw=True)

    actual_days = close.rolling(
        window=rolling_returns_bars,
        min_periods=min_periods,
    ).apply(lambda x: len(x), raw=True)

    period_return = close / first_price - 1
    annualised = (1 + period_return) ** (365 / actual_days) - 1
    return annualised



@indicators.define()
def rolling_volatility(
    close: pd.Series,
    volatility_window: int = 180,
) -> pd.Series:
    """Calculate annualised rolling volatility.

    - Use last 6 months (180 days) of daily price data
    - If fewer than 180 days available, annualise whatever exists
    - Require minimum 14 days of data
    """
    min_periods = 14
    daily_returns = close.pct_change()
    rolling_std = daily_returns.rolling(
        window=volatility_window,
        min_periods=min_periods,
    ).std()
    annualised = rolling_std * (365 ** 0.5)
    return annualised


@indicators.define()
def rolling_sharpe(
    close: pd.Series,
    volatility_window: int = 180,
) -> pd.Series:
    """Calculate rolling Sharpe ratio.

    - Annualised return / annualised volatility
    - Uses volatility_window for lookback period
    - Requires minimum 14 days of data
    """
    min_periods = 14
    daily_returns = close.pct_change()

    rolling_mean = daily_returns.rolling(
        window=volatility_window,
        min_periods=min_periods,
    ).mean()

    rolling_std = daily_returns.rolling(
        window=volatility_window,
        min_periods=min_periods,
    ).std()

    # Annualise: mean * 365 / (std * sqrt(365)) = mean * sqrt(365) / std
    sharpe = (rolling_mean * 365) / (rolling_std * (365 ** 0.5))
    return sharpe


@indicators.define()
def rolling_sortino(
    close: pd.Series,
    volatility_window: int = 180,
) -> pd.Series:
    """Calculate rolling Sortino ratio.

    - Annualised return / annualised downside deviation
    - Uses volatility_window for lookback period
    - Requires minimum 14 days of data
    """
    min_periods = 14
    daily_returns = close.pct_change()

    rolling_mean = daily_returns.rolling(
        window=volatility_window,
        min_periods=min_periods,
    ).mean()

    # Downside deviation: std of negative returns only (clipped at 0)
    downside_returns = daily_returns.clip(upper=0)
    rolling_downside_std = downside_returns.rolling(
        window=volatility_window,
        min_periods=min_periods,
    ).std()

    # Annualise
    sortino = (rolling_mean * 365) / (rolling_downside_std * (365 ** 0.5))
    return sortino

@indicators.define(source=IndicatorSource.tvl)
def tvl(
    close: pd.Series,
    execution_context: ExecutionContext,
    timestamp: pd.Timestamp,
) -> pd.Series:
    """Get TVL series for a pair.

    - Because TVL data is 1d and we use 1h everywhere else, we need to forward fill

    - Use previous hourly close as the value
    """
    if execution_context.live_trading:
        # TVL is daily data.
        # We need to forward fill until the current hour.
        # Use our special ff function.        
        assert isinstance(timestamp, pd.Timestamp), f"Live trading needs forward-fill end time, we got {timestamp}"
        from tradingstrategy.utils.forward_fill import forward_fill
        df = pd.DataFrame({"close": close})
        df_ff = forward_fill(
            df,
            Parameters.candle_time_bucket.to_frequency(),
            columns=("close",),
            forward_fill_until=timestamp,
        )
        series = df_ff["close"]
        return series
    else:
        return close.resample("1h").ffill()


@indicators.define(dependencies=(tvl,), source=IndicatorSource.dependencies_only_universe)
def tvl_inclusion_criteria(   
    min_tvl: USDollarAmount,
    dependency_resolver: IndicatorDependencyResolver,
) -> pd.Series:
    """The pair must have min XX,XXX USD one-sided TVL to be included.

    - If the Uniswap pool does not have enough ETH or USDC deposited, skip the pair as a scam

    :return:
        Series where each timestamp is a list of pair ids meeting the criteria at that timestamp
    """
    
    series = dependency_resolver.get_indicator_data_pairs_combined(tvl)
    mask = series >= min_tvl
    # Turn to a series of lists
    mask_true_values_only = mask[mask == True]
    series = mask_true_values_only.groupby(level='timestamp').apply(lambda x: x.index.get_level_values('pair_id').tolist())
    return series



@indicators.define(
    source=IndicatorSource.strategy_universe
)
def trading_availability_criteria(
    strategy_universe: TradingStrategyUniverse,
) -> pd.Series:
    """Is pair tradeable at each hour.

    - The pair has a price candle at that
    - Mitigates very corner case issues that TVL/liquidity data is per-day whileas price data is natively per 1h
      and the strategy inclusion criteria may include pair too early hour based on TVL only,
      leading to a failed attempt to rebalance in a backtest
    - Only relevant for backtesting issues if we make an unlucky trade on the starting date
      of trading pair listing

    :return:
        Series with with index (timestamp) and values (list of pair ids trading at that hour)
    """
    # Trading pair availability is defined if there is a open candle in the index for it.
    # Because candle data is forward filled, we should not have any gaps in the index.
    candle_series = strategy_universe.data_universe.candles.df["open"]
    pairs_per_timestamp = candle_series.groupby(level='timestamp').apply(lambda x: x.index.get_level_values('pair_id').tolist())
    return pairs_per_timestamp


@indicators.define(
    dependencies=[
        tvl_inclusion_criteria,
        trading_availability_criteria
    ],
    source=IndicatorSource.strategy_universe
)
def inclusion_criteria(
    strategy_universe: TradingStrategyUniverse,
    min_tvl: USDollarAmount,
    dependency_resolver: IndicatorDependencyResolver
) -> pd.Series:
    """Pairs meeting all of our inclusion criteria.

    - Give the tradeable pair set for each timestamp

    :return:
        Series where index is timestamp and each cell is a list of pair ids matching our inclusion criteria at that moment
    """

    # Filter out benchmark pairs like WETH in the tradeable pair set
    benchmark_pair_ids = set(strategy_universe.get_pair_by_human_description(desc).internal_id for desc in SUPPORTING_PAIRS)

    tvl_series = dependency_resolver.get_indicator_data(
        tvl_inclusion_criteria,
        parameters={
            "min_tvl": min_tvl,
        },
    )

    trading_availability_series = dependency_resolver.get_indicator_data(trading_availability_criteria)

    #
    # Process all pair ids as a set and the final inclusion
    # criteria is union of all sub-criterias
    #

    df = pd.DataFrame({
        "tvl_pair_ids": tvl_series,
        "trading_availability_pair_ids": trading_availability_series,
    })

    # https://stackoverflow.com/questions/33199193/how-to-fill-dataframe-nan-values-with-empty-list-in-pandas
    df = df.fillna("").apply(list)

    def _combine_criteria(row):
        final_set = set(row["tvl_pair_ids"]) & \
                    set(row["trading_availability_pair_ids"])
        return final_set - benchmark_pair_ids

    union_criteria = df.apply(_combine_criteria, axis=1)

    # Inclusion criteria data can be spotty at the beginning when there is only 0 or 1 pairs trading,
    # so we need to fill gaps to 0
    full_index = pd.date_range(
        start=union_criteria.index.min(),
        end=union_criteria.index.max(),
        freq=Parameters.candle_time_bucket.to_frequency(),
    )
    reindexed = union_criteria.reindex(full_index, fill_value=[])
    return reindexed


@indicators.define(dependencies=(tvl_inclusion_criteria,), source=IndicatorSource.dependencies_only_universe)
def tvl_included_pair_count(
        min_tvl: USDollarAmount,
        dependency_resolver: IndicatorDependencyResolver
) -> pd.Series:
    """Calculate number of pairs in meeting volatility criteria on each timestamp"""
    series = dependency_resolver.get_indicator_data(
        tvl_inclusion_criteria,
        parameters={"min_tvl": min_tvl},
    )
    series = series.apply(len)

    # TVL data can be spotty at the beginning when there is only 0 or 1 pairs trading,
    # so we need to fill gaps to 0
    full_index = pd.date_range(
        start=series.index.min(),
        end=series.index.max(),
        freq=Parameters.candle_time_bucket.to_frequency(),
    )
    # Reindex and fill NaN with zeros
    reindexed = series.reindex(full_index, fill_value=0)
    return reindexed


@indicators.define(dependencies=(inclusion_criteria,), source=IndicatorSource.dependencies_only_universe)
def all_criteria_included_pair_count(
    min_tvl: USDollarAmount,
    dependency_resolver: IndicatorDependencyResolver
) -> pd.Series:
    """Series where each timestamp is the list of pairs meeting all inclusion criteria.

    :return:
        Series with pair count for each timestamp
    """
    series = dependency_resolver.get_indicator_data(
        "inclusion_criteria",
        parameters={
            "min_tvl": min_tvl, 
        },
    )
    return series.apply(len)


@indicators.define(source=IndicatorSource.strategy_universe)
def trading_pair_count(
    strategy_universe: TradingStrategyUniverse,
) -> pd.Series:
    """Get number of pairs that trade at each timestamp.

    - Pair must have had at least one candle before the timestamp to be included

    - Exclude benchmarks pairs we do not trade

    :return:
        Series with pair count for each timestamp
    """

    benchmark_pair_ids = {strategy_universe.get_pair_by_human_description(desc).internal_id for desc in SUPPORTING_PAIRS}

    # Get pair_id, timestamp -> timestamp, pair_id index
    series = strategy_universe.data_universe.candles.df["open"]    
    swap_index = series.index.swaplevel(0, 1)

    seen_pairs = set()
    seen_data = {}

    for timestamp, pair_id in swap_index:
        if pair_id in benchmark_pair_ids:
            continue
        seen_pairs.add(pair_id)
        seen_data [timestamp] = len(seen_pairs)

    series = pd.Series(seen_data.values(), index=list(seen_data.keys()))
    return series





display_indicators(indicators)



# Time range for backtest

- Choose the backtesting time range
- Start when we have enough assets (`Parameters.min_asset_universe`) in our asset universe to form the first basket

In [ ]:
backtest_start = Parameters.backtest_start
backtest_end = Parameters.backtest_end

print(f"Time range is {backtest_start} - {backtest_end}")


# Algorithm and backtest

- Run the backtest

In [ ]:
import numpy as np

from tradeexecutor.backtest.backtest_runner import run_backtest_inline
from tradeexecutor.strategy.alpha_model import AlphaModel
from tradeexecutor.state.trade import TradeExecution
from tradeexecutor.strategy.pandas_trader.strategy_input import StrategyInput, IndicatorDataNotFoundWithinDataTolerance
from tradeexecutor.state.visualisation import PlotKind
from tradeexecutor.backtest.backtest_runner import run_backtest_inline
from tradeexecutor.strategy.tvl_size_risk import USDTVLSizeRiskModel
from tradeexecutor.strategy.weighting import weight_by_1_slash_n, weight_by_1_slash_signal, weight_passthrouh, weight_equal, weight_by_log
from tradeexecutor.utils.dedent import dedent_any
from tradeexecutor.strategy.pandas_trader.yield_manager import YieldManager, YieldRuleset, YieldWeightingRule, YieldDecisionInput
from tradeexecutor.strategy.execution_context import ExecutionContext, ExecutionMode



def decide_trades(
    input: StrategyInput
) -> list[TradeExecution]:
    """For each strategy tick, generate the list of trades."""
    parameters = input.parameters
    position_manager = input.get_position_manager()
    state = input.state
    timestamp = input.timestamp
    indicators = input.indicators
    strategy_universe = input.strategy_universe

    portfolio = position_manager.get_current_portfolio()
    equity = portfolio.get_total_equity()
    
    # All gone, stop doing decisions
    if input.execution_context.mode == ExecutionMode.backtesting:
        if equity < parameters.initial_cash * 0.10:
            return []
            
    # Build signals for each pair
    alpha_model = AlphaModel(
        timestamp,
        close_position_weight_epsilon=parameters.min_portfolio_weight,  # 10 BPS is our min portfolio weight
    )

    tvl_included_pair_count = indicators.get_indicator_value(
        "tvl_included_pair_count",
    )

    # Get pairs included in this rebalance cycle.
    # This includes pair that have been pre-cleared in inclusion_criteria()
    # with volume, volatility and TVL filters
    included_pairs = indicators.get_indicator_value(
        "inclusion_criteria",
        na_conversion=False,
    )
    if included_pairs is None:
        included_pairs = []

    # Determine which indicator to use as the signal based on weighting_method
    weighting_method = parameters.weighting_method
    if weighting_method == "rolling_returns" or weighting_method == "inverse_volatility":
        signal_indicator_name = "rolling_returns"
    elif weighting_method == "rolling_sharpe":
        signal_indicator_name = "rolling_sharpe"
    elif weighting_method == "rolling_sortino":
        signal_indicator_name = "rolling_sortino"

    # Set signal for each pair
    signal_count = 0
    pair_volatilities = {}

    for pair_id in included_pairs:
        pair = strategy_universe.get_pair_by_id(pair_id)

        if not state.is_good_pair(pair):
            # Tradeable flag set to False, etc.
            continue

        pair_signal = indicators.get_indicator_value(signal_indicator_name, pair=pair)
        if pair_signal is None or pair_signal < 0:
            continue

        # For inverse volatility, also collect rolling volatility per pair
        if weighting_method == "inverse_volatility":
            pair_vol = indicators.get_indicator_value("rolling_volatility", pair=pair)
            if pair_vol is None or pair_vol <= 0:
                continue
            pair_volatilities[pair.internal_id] = pair_vol

        alpha_model.set_signal(
            pair,
            pair_signal,
        )

        # Diagnostics reporting
        signal_count += 1

    # Calculate how much dollar value we want each individual position to be on this strategy cycle,
    # based on our total available equity
    portfolio = position_manager.get_current_portfolio()
    equity = portfolio.get_total_equity()
    portfolio_target_value = equity * parameters.allocation

    # We are greedy and take as many positions needed until we run out of investable equity
    alpha_model.select_top_signals(
        count=999,
    )

    # Assign weights based on weighting method
    if weighting_method == "inverse_volatility":
        # Override signal values with volatility for inverse vol weighting
        # select_top_signals picks by rolling returns, assign_weights uses 1/volatility
        for pair_id, signal_obj in alpha_model.signals.items():
            if pair_id in pair_volatilities:
                signal_obj.signal = pair_volatilities[pair_id]
        alpha_model.assign_weights(method=weight_by_1_slash_signal)
    else:
        # Resolve weight function from parameter
        weight_func_map = {
            "weight_equal": weight_equal,
            "weight_1_slash_n": weight_by_1_slash_n,
        }
        weight_func = weight_func_map[parameters.weight_function]
        alpha_model.assign_weights(method=weight_func)

    #
    # Normalise weights and cap the positions
    #
    size_risk_model = USDTVLSizeRiskModel(
        pricing_model=input.pricing_model,
        per_position_cap=parameters.per_position_cap_of_pool,  # This is how much % by all pool TVL we can allocate for a position
        missing_tvl_placeholder_usd=0.0,  # Placeholder for missing TVL data until we get the data off the chain
    )

    # Use higher concentration when there are not enough pairs in the cycle
    # to avoid sitting on unused cash during early backtest periods
    if len(included_pairs) < parameters.early_allocation_vault_count_threshold:
        max_weight = parameters.early_allocation_max_contration / 100
    else:
        max_weight = float(parameters.max_concentration)

    alpha_model.normalise_weights(
        investable_equity=portfolio_target_value,
        size_risk_model=size_risk_model,
        max_weight=max_weight,
        max_positions=parameters.max_assets_in_portfolio,
        waterfall=True,
    )

    # Load in old weight for each trading pair signal,
    # so we can calculate the adjustment trade size
    alpha_model.update_old_weights(
        state.portfolio,
        ignore_credit=False,
    )
    alpha_model.calculate_target_positions(position_manager)

    # Shift portfolio from current positions to target positions
    # determined by the alpha signals (momentum)

    # rebalance_threshold_usd = portfolio_target_value * parameters.min_rebalance_trade_threshold_pct
    rebalance_threshold_usd = parameters.individual_rebalance_min_threshold_usd

    assert rebalance_threshold_usd > 0.1, "Safety check tripped - something like wrong with strat code"
    trades = alpha_model.generate_rebalance_trades_and_triggers(
        position_manager,
        min_trade_threshold=rebalance_threshold_usd,  # Don't bother with trades under XXXX USD
        invidiual_rebalance_min_threshold=parameters.individual_rebalance_min_threshold_usd,
        sell_rebalance_min_threshold=parameters.sell_rebalance_min_threshold,
        execution_context=input.execution_context,
    )
        
    # Add verbal report about decision made/not made,
    # so it is much easier to diagnose live trade execution.
    # This will be readable in Discord/Telegram logging.
    if input.is_visualisation_enabled():
        try:
            top_signal = next(iter(alpha_model.get_signals_sorted_by_weight()))
            if top_signal.normalised_weight == 0:
                top_signal = None
        except StopIteration:
            top_signal = None

        rebalance_volume = sum(t.get_value() for t in trades)

        report = dedent_any(f"""
        Cycle: #{input.cycle}
        Rebalanced: {'👍' if alpha_model.is_rebalance_triggered() else '👎'}
        Open/about to open positions: {len(state.portfolio.open_positions)} 
        Max position value change: {alpha_model.max_position_adjust_usd:,.2f} USD
        Rebalance threshold: {alpha_model.position_adjust_threshold_usd:,.2f} USD
        Trades decided: {len(trades)}
        Pairs total: {strategy_universe.data_universe.pairs.get_count()}
        Pairs meeting inclusion criteria: {len(included_pairs)}
        Pairs meeting TVL inclusion criteria: {tvl_included_pair_count}        
        Signals created: {signal_count}
        Total equity: {portfolio.get_total_equity():,.2f} USD
        Cash: {position_manager.get_current_cash():,.2f} USD
        Investable equity: {alpha_model.investable_equity:,.2f} USD
        Accepted investable equity: {alpha_model.accepted_investable_equity:,.2f} USD
        Allocated to signals: {alpha_model.get_allocated_value():,.2f} USD
        Discarted allocation because of lack of lit liquidity: {alpha_model.size_risk_discarded_value:,.2f} USD
        Rebalance volume: {rebalance_volume:,.2f} USD
        """)

        # Most volatility pair signal weight (normalised): {max_vol_signal.normalised_weight * 100 if max_vol_signal else '-'} % (got {max_vol_signal.position_size_risk.get_relative_capped_amount() * 100 if max_vol_signal else '-'} % of asked size)
        if top_signal:
            assert top_signal.position_size_risk
            report += dedent_any(f"""
            Top signal pair: {top_signal.pair.get_ticker()}
            Top signal value: {top_signal.signal}
            Top signal weight: {top_signal.raw_weight}
            Top signal weight (normalised): {top_signal.normalised_weight * 100:.2f} % (got {top_signal.position_size_risk.get_relative_capped_amount() * 100:.2f} % of asked size)
            """)

        for flag, count in alpha_model.get_flag_diagnostics_data().items():
            report += f"Signals with flag {flag.name}: {count}\n"

        state.visualisation.add_message(
            timestamp,
            report,
        )

        state.visualisation.set_discardable_data("alpha_model", alpha_model)

    return trades  # Return the list of trades we made in this cycle


# Optimiser

- Run optimiser

In [ ]:
import logging
from tradeexecutor.backtest.optimiser import perform_optimisation
from tradeexecutor.backtest.optimiser import prepare_optimiser_parameters
from tradeexecutor.backtest.optimiser import MinTradeCountFilter
from tradeexecutor.backtest.optimiser_functions import optimise_sharpe, optimise_sortino, optimise_profit

# How many Gaussian Process iterations we do
iterations = 16

# What do we optimise for
# search_func = BalancedSharpeAndMaxDrawdownOptimisationFunction(sharpe_weight=0.75, max_drawdown_weight=0.25)
search_func = optimise_sharpe

optimiser_result = perform_optimisation(
    iterations=iterations,
    search_func=search_func,
    decide_trades=decide_trades,
    strategy_universe=strategy_universe,
    parameters=prepare_optimiser_parameters(Parameters),  # Handle scikit-optimise search space
    create_indicators=indicators.create_indicators,
    result_filter=MinTradeCountFilter(50),
    timeout=70*60,    
    batch_size=5,
    # We are searching wacky parameter combinations and
    # some of them lead to buggy strategies,
    # by setting ignore_wallet_errors we just zero out buggy
    # strategies instead of crashing with an exception
    ignore_wallet_errors=True,  
    # Uncomment for diagnostics
    # log_level=logging.INFO,
    # max_workers=1,
)

print(f"Optimise completed, optimiser searched {optimiser_result.get_combination_count()} combinations, with {optimiser_result.get_cached_count()} results read directly from cache. and {optimiser_result.get_filtered_count()} filtered results.")
print("Backtests failed with exception:", optimiser_result.get_failed_count())

# Results

- Show the top results of all optimiser iterations in a single table

In [ ]:
from tradeexecutor.analysis.optimiser import analyse_optimiser_result
from tradeexecutor.analysis.grid_search import render_grid_search_result_table


filtered = [r for r in optimiser_result.results if r.filtered]
print(f"Filtering out {len(filtered)} results")

# Optimiser already filtered for min_positions_threshold when doing the optimiser run
df = analyse_optimiser_result(
    optimiser_result,
    max_search_results=300,
    drop_duplicates=False,
)
print(f"Showing the best {len(df)} results")

render_grid_search_result_table(df)
 

# Parameter analysis

## Decision tree visualisation

In [ ]:
import matplotlib.pyplot as plt

from sklearn.calibration import LabelEncoder
from sklearn.tree import DecisionTreeRegressor, plot_tree

parameter_names = list(df.index.names)
parameter_df = df
parameter_df = parameter_df.reset_index()

analysis_metric = "CAGR"
print(f"Analysing {analysis_metric} for parameters: {parameter_names}")

# Prepare data
X = parameter_df[parameter_names]
y = parameter_df[analysis_metric]

# Convert categorical values if needed
for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])

# Train a simple decision tree
tree = DecisionTreeRegressor(max_depth=4)
tree.fit(X, y)

# Plot
plt.figure(figsize=(20, 10))
plot_tree(tree, feature_names=X.columns, filled=True, fontsize=10)
plt.title('Decision Tree for Parameter Importance')
plt.show()

## Feature importance analysis

Use a regression model to determine which parameters have the strongest influence.



In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import numpy as np

# Prepare data
X = parameter_df[parameter_names]
y = parameter_df[analysis_metric]

# Convert categorical values to numeric if needed
for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])

# Train a random forest model
model = RandomForestRegressor(n_estimators=100)
model.fit(X, y)

# Plot feature importances
importances = model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10, 6))
plt.title(f'Feature Importances for {analysis_metric}')
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

## Heatmaps for parameter pairs

In [ ]:
import plotly.graph_objects as go
import numpy as np

def create_heatmap(df, param1, param2, metric):
    # Create pivot table
    pivot = df.pivot_table(
        values=metric, 
        index=param1,
        columns=param2,
        aggfunc=np.mean
    )
    
    fig = go.Figure(data=go.Heatmap(
        z=pivot.values,
        x=pivot.columns,
        y=pivot.index,
        colorscale='Viridis',
        colorbar=dict(title=metric)
    ))
    
    fig.update_layout(
        title=f'Impact of {param1} and {param2} on {metric}',
        xaxis_title=param2,
        yaxis_title=param1
    )
    
    return fig

# Example parameter pairs to analyze
param_pairs = []
for i in range(1, len(parameter_names), 1):
    param_pairs.append((parameter_names[i], parameter_names[i-1]))

for param1, param2 in param_pairs:
    fig = create_heatmap(parameter_df, param1, param2, metric=analysis_metric)
    fig.show()

## Cluster analysis

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# Select features for clustering
features = parameter_names
X = parameter_df[features]

# Convert categorical values to numeric
for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply K-means clustering (determine optimal k separately)
kmeans = KMeans(n_clusters=5, random_state=42)
parameter_df['cluster'] = kmeans.fit_predict(X_scaled)

# Analyze performance by cluster
cluster_perf = parameter_df.groupby('cluster').agg({
    'CAGR': ['mean', 'min', 'max', 'count'],
    'Sharpe': ['mean', 'min', 'max'],
    'Max DD': ['mean', 'min', 'max']
}).reset_index()

display(cluster_perf)

# Visualize clusters in 3D space (using first 3 PCA components)
from sklearn.decomposition import PCA

try:
    pca = PCA(n_components=3)
    components = pca.fit_transform(X_scaled)

    fig = px.scatter_3d(
        x=components[:, 0],
        y=components[:, 1],
        z=components[:, 2],
        color=parameter_df['cluster'],
        hover_data={'Sharpe': parameter_df['Sharpe'], 'CAGR': parameter_df['CAGR']},
        title='Clustering of Parameter Combinations'
    )
    fig.show()
except ValueError as e:
    print(f"Error in PCA: {e}")
    print("PCA failed, possibly due to too few features or singular matrix.")
    # Handle the error as needed

## Parallel coordinates plot

In [ ]:
import plotly.express as px

# Extract top results (e.g., top 50)
top_results = parameter_df.head(50)

dimensions = parameter_names + [analysis_metric]

# Create parallel coordinates plot
fig = px.parallel_coordinates(
    parameter_df,
    dimensions=dimensions,
    color=analysis_metric,
    color_continuous_scale=px.colors.sequential.Peach,
    title='Parameter Impact on Strategy Performance'
)


# Increase font size for axis titles, tick labels, and title
fig.update_layout(
    font=dict(size=14),  # Base font size for the whole figure
    title=dict(
        text='Parameter Impact on Strategy Performance',
        font=dict(size=20)  # Title font size
    ),
    coloraxis_colorbar=dict(
        title=dict(
            text=analysis_metric,
            font=dict(size=16)
        ),
        tickfont=dict(size=14)
    )
)

# Parallel coordinates chart: Line width not supported
# https://github.com/plotly/plotly.js/issues/2573

fig.show()

# The best candidate equity curve

In [ ]:
from tradeexecutor.visual.grid_search import visualise_single_grid_search_result_benchmark

# GridSearchResult instance that gave the best performance
best_pick = optimiser_result.results[0].result
state = best_pick.hydrate_state()

print(f"The best result found for {search_func} was {best_pick}")

fig = visualise_single_grid_search_result_benchmark(
    best_pick, 
    strategy_universe, 
    initial_cash=Parameters.initial_cash,
    log_y=True,
)
fig.show()

# Portfolio performance (best pick)

- Compare buy and hold against our best performance

In [ ]:
from tradeexecutor.analysis.multi_asset_benchmark import compare_strategy_backtest_to_multiple_assets

returns = best_pick.returns

df = compare_strategy_backtest_to_multiple_assets(
    state=state,
    strategy_universe=strategy_universe,
    returns=returns,
    display=True,
    asset_count=3,
)
display(df)

# Trade summary (best pick)

- Show statistics about the made trades

In [ ]:
summary = best_pick.summary
display(summary.to_dataframe())

# Trading pair performance breakdown

- Show breakdown of different pairs on the best result

In [ ]:
from tradeexecutor.analysis.multipair import analyse_multipair
from tradeexecutor.analysis.multipair import format_multipair_summary

multipair_summary = analyse_multipair(state)
display(format_multipair_summary(multipair_summary))

# Best positions

- Find positions that made most profit for the leading backtest

In [ ]:
profit_by_position = [(p.get_realised_profit_usd(), p) for p in state.portfolio.get_all_positions()]
profit_by_position.sort(key=lambda t: t[0] or -999_999_999, reverse=True)

data = []

for profit, p in profit_by_position[0:5]:
    data.append({
        "Profit USD": profit,
        "Pair": p.pair.get_ticker(),
        "Position id": p.position_id,
        "Opened": p.opened_at,
        "Closed": p.closed_at,
        "Duration": p.closed_at - p.opened_at if p.closed_at else "(still open)",
        "Open price": p.get_opening_price(),
        "Close price": p.get_closing_price(),
        "Trades": p.get_trade_count(),
        "Price gain %": (p.get_closing_price() - p.get_opening_price()) / p.get_opening_price() * 100,
        "Weight at open %": p.get_capital_tied_at_open_pct() * 100,
    })

df = pd.DataFrame(data)
display(df)

# Rolling sharpe

- The rolling sharpe ratio of the best pick

In [ ]:
import plotly.express as px

from tradeexecutor.visual.equity_curve import calculate_equity_curve, calculate_returns
from tradeexecutor.visual.equity_curve import calculate_rolling_sharpe

rolling_sharpe = calculate_rolling_sharpe(
    returns,
    freq="D",
    periods=180,
)

fig = px.line(rolling_sharpe, title='Strategy rolling Sharpe (6 months)')
fig.update_layout(showlegend=False)
fig.update_yaxes(title="Sharpe")
fig.update_xaxes(title="Time")
fig.show()

# Data diagnosics

- Some example code to track down issues with data


In [ ]:
# data = [{"Timestamp": s.calculated_at, "Equity": s.net_asset_value} for s in state.stats.portfolio]
# df = pd.DataFrame(data)
# df = df.set_index("Timestamp")

# with pd.option_context('display.min_rows', 50, "display.max_rows", 50):
#     display(df)

In [ ]:
# for t in state.portfolio.get_all_trades():
#     if t.executed_at.date() == datetime.date(2023, 12, 29):
#         print(t, t.pair, t.get_value())

In [ ]:
# import plotly.graph_objects as go

# pair_desc = (ChainId.ethereum, "uniswap-v3", "WBTC", "USDC", 0.0005)
# pair = strategy_universe.get_pair_by_human_description(pair_desc)
# df = strategy_universe.data_universe.candles.get_candles_by_pair(pair.internal_id)


# fig = go.Figure(data=go.Ohlc(x=df.timestamp,
#                     open=df.open,
#                     high=df.high,
#                     low=df.low,
#                     close=df.close))
# fig.show()

# with pd.option_context('display.min_rows', 5000, "display.max_rows", 5000, 'display.float_format', lambda x: f'{x:.6f}'):
#     display(df)